# Unified Classical AI Reasoning Agent

This notebook implements one prediction interface with three specialized classical reasoning modules: Horn-clause reasoning for RuleTaker, first-order resolution for FOLIO, and structural causal model reasoning for CounterBench. Dataset names are not required by the prediction function. The public evaluation removes source labels before each prediction and checks automatic routing.

The private-test CSV is not generated from public examples. The final cell contains the instruction for the evaluator to supply the private file when available.


# Classical Reasoning Agent: Natural Language KB to True/False

**Pipeline:** natural-language context and query -> parser -> internal representation -> classical reasoning engine -> `True` / `False`

| Stage | Technique |
|---|---|
| Representation | Function-free Horn clauses for facts and rules; first-order formula trees and CNF clauses for FOLIO; structural causal equations for CounterBench |
| Horn reasoning | Semi-naive forward chaining, unification, argument indexing, stratified negation-as-failure, and backward chaining as an alternative engine |
| First-order reasoning | Quantifier handling, Skolemization, CNF conversion, unification, binary resolution, factoring and proof search |
| Causal reasoning | Structural causal models, abduction over exogenous variables, interventions and forward simulation |
| Routing | Input-schema and content-based routing; the predictor does not require a dataset name |

No generative model is used to decide whether a query is True or False. The final predictions are produced by the symbolic reasoning engines.


## 1. Setup, constants and text utilities

In [1]:
import os
import re
import sys
import csv
import json
import time
import heapq
import random
import itertools
from collections import defaultdict, deque, Counter

NEG_PREFIX = "~"
RESOLUTION_MAX_CLAUSES = 8000
RESOLUTION_MAX_SECONDS = 2.0
RESOLUTION_MAX_WEIGHT = 60
CNF_MAX_CLAUSES = 2000
BACKWARD_MAX_DEPTH = 40
BACKWARD_MAX_NODES = 30000
RANDOM_SEED = 13
random.seed(RANDOM_SEED)

ARTICLES = {"the", "a", "an"}
COPULAS = {"is", "are", "was", "were", "be"}
DO_AUX = {"does", "do", "did"}
MODAL_VERBS = {"can", "could", "will", "would", "must", "should", "may", "might"}
HAVE_FORMS = {"has", "have", "had"}
PREPOSITIONS = {"to", "of", "with", "by", "for", "in", "on", "at", "from", "about", "into", "near", "under", "over", "between", "among"}
INDEFINITE_PRONOUNS = {"someone", "something", "somebody", "anyone", "anything", "anybody", "everyone", "everything", "everybody"}
ANAPHORA = {"they", "it", "them"}
GENERIC_NOUNS = {"thing", "person", "one", "animal"}
STRICT_REJECT = {"that", "who", "which", "whom", "whose", "neither", "nor", "either", "only", "than", "both", "most", "least", "some", "no", "unless", "every", "all", "any", "each", "if", "then", "or", "and", "because", "but", "when", "while", "there", "also"}

CONTRACTIONS = [
    (r"\bdoesn't\b", "does not"),
    (r"\bdon't\b", "do not"),
    (r"\bdidn't\b", "did not"),
    (r"\bisn't\b", "is not"),
    (r"\baren't\b", "are not"),
    (r"\bwasn't\b", "was not"),
    (r"\bweren't\b", "were not"),
    (r"\bhasn't\b", "has not"),
    (r"\bhaven't\b", "have not"),
    (r"\bcan't\b", "can not"),
    (r"\bcannot\b", "can not"),
    (r"\bwon't\b", "will not"),
]

IRREGULAR_SINGULAR = {
    "people": "person", "men": "man", "women": "woman", "children": "child",
    "has": "have", "does": "do", "mice": "mouse", "geese": "goose",
    "wolves": "wolf", "knives": "knife", "lives": "life", "feet": "foot",
    "teeth": "tooth", "ones": "one",
}


In [2]:
def normalize_sentence(text):
    t = str(text).strip()
    t = t.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    t = t.lower()
    for pattern, replacement in CONTRACTIONS:
        t = re.sub(pattern, replacement, t)
    t = re.sub(r"\s+", " ", t)
    return t.strip().rstrip(".!?;: ").strip()


def split_sentences(text):
    if text is None:
        return []
    if isinstance(text, (list, tuple)):
        text = "\n".join(str(x) for x in text)
    sentences = []
    for line in str(text).replace("\r", "\n").split("\n"):
        line = line.strip()
        if not line:
            continue
        for part in re.split(r"(?<=[.!?])\s+", line):
            part = part.strip()
            if part:
                sentences.append(part)
    return sentences


def singular(word):
    if word in IRREGULAR_SINGULAR:
        return IRREGULAR_SINGULAR[word]
    if len(word) > 4 and word.endswith("ies"):
        return word[:-3] + "y"
    if len(word) > 4 and word.endswith(("ches", "shes", "sses", "xes", "zes")):
        return word[:-2]
    if len(word) > 3 and word.endswith("s") and not word.endswith(("ss", "us", "is")):
        return word[:-1]
    return word


def is_variable(term):
    return isinstance(term, str) and term[:1] == "?"
try:
    import pandas as pd
except ImportError:
    pd = None

from pathlib import Path


## 2. Natural language to Horn clauses (RuleTaker style)

In [3]:
class VarContext:
    def __init__(self):
        self.names = {}

    def subject_var(self):
        if "x" not in self.names:
            self.names["x"] = "?x"
        return self.names["x"]

    def object_var(self):
        if "x" not in self.names:
            self.names["x"] = "?x"
            return "?x"
        if "y" not in self.names:
            self.names["y"] = "?y"
        return self.names["y"]


def make_entity(tokens, ctx, role):
    words = [t for t in tokens if t not in ARTICLES]
    if not words:
        return None
    if len(words) == 1 and tokens and tokens[0] in ("a", "an", "any") and singular(words[0]) in ("person", "thing", "one"):
        return ctx.subject_var() if role == "subject" else ctx.object_var()
    if len(words) == 1:
        w = words[0]
        if w in INDEFINITE_PRONOUNS:
            return ctx.subject_var() if role == "subject" else ctx.object_var()
        if w in ANAPHORA:
            return ctx.subject_var()
    return "_".join(words)


def parse_atomic_ex(sentence, ctx):
    toks = [t.strip(",") for t in sentence.split()]
    toks = [t for t in toks if t]
    if not toks or len(toks) > 12:
        return None
    if any(t in STRICT_REJECT for t in toks):
        return None
    cop = next((i for i, t in enumerate(toks) if t in COPULAS), None)
    if cop is not None:
        if cop == 0:
            return None
        subject_tokens = toks[:cop]
        rest = toks[cop + 1:]
        negated = False
        if rest and rest[0] == "not":
            negated = True
            rest = rest[1:]
        if not rest:
            return None
        subject = make_entity(subject_tokens, ctx, "subject")
        if subject is None:
            return None
        prep = next((i for i, t in enumerate(rest) if t in PREPOSITIONS and i > 0), None)
        if prep is not None:
            obj = make_entity(rest[prep + 1:], ctx, "object")
            if obj is None:
                return None
            pred = "_".join(w for w in rest[:prep + 1] if w not in ARTICLES)
            return (negated, pred, (subject, obj)), " ".join(subject_tokens)
        words = [w for w in rest if w not in ARTICLES]
        if not words:
            return None
        return (negated, "_".join(words), (subject,)), " ".join(subject_tokens)
    work = list(toks)
    negated = False
    if "not" in work:
        n = work.index("not")
        if n >= 1 and work[n - 1] in DO_AUX:
            negated = True
            work = work[:n - 1] + work[n + 1:]
        else:
            return None
    positions = [i for i, t in enumerate(work) if t == "the" and i > 0]
    if positions:
        k = positions[-1]
        object_tokens = work[k:]
        verb_index = k - 1
    else:
        object_tokens = work[-1:]
        verb_index = len(work) - 2
    if verb_index < 1:
        return None
    verb = work[verb_index]
    if verb in ARTICLES or not verb.isalpha():
        return None
    subject_tokens = work[:verb_index]
    subject = make_entity(subject_tokens, ctx, "subject")
    obj = make_entity(object_tokens, ctx, "object")
    if subject is None or obj is None:
        return None
    return (negated, singular(verb), (subject, obj)), " ".join(subject_tokens)


In [4]:
def parse_conjunction(text, ctx):
    pieces = [p.strip() for p in re.split(r"\s*,?\s+and\s+", text) if p.strip()]
    if not pieces:
        return None
    literals = []
    previous_subject = None
    for piece in pieces:
        words = piece.split()
        bare = previous_subject is not None and (len(words) == 1 or (len(words) == 2 and words[0] == "not"))
        parsed = parse_atomic_ex((previous_subject + " is " + piece) if bare else piece, ctx)
        if parsed is None:
            return None
        literal, subject_surface = parsed
        previous_subject = subject_surface
        literals.append(literal)
    return literals


def parse_if_rule(sentence):
    m = re.match(r"^if (.+?),? then (.+)$", sentence)
    if not m:
        return None
    condition, conclusion = m.group(1), m.group(2)
    ctx = VarContext()
    alternatives = []
    for alternative in re.split(r"\s+or\s+", condition):
        literals = parse_conjunction(alternative, ctx)
        if literals is None:
            return None
        alternatives.append(literals)
    heads = parse_conjunction(conclusion, ctx)
    if heads is None:
        return None
    return [(list(body), head) for body in alternatives for head in heads]


def parse_universal(sentence):
    m = re.match(r"^(?:(all|every|each|any)\s+)?(.+?)\s+(are|is)\s+(not\s+)?(.+)$", sentence)
    if not m:
        return None
    quantifier, subject, _, negation, predicate = m.groups()
    words = [w for w in re.split(r"[,\s]+", subject) if w and w != "and"]
    if not words:
        return None
    last = singular(words[-1])
    generic = last in GENERIC_NOUNS
    if not generic and quantifier is None:
        return None
    if any(w in STRICT_REJECT or w in ARTICLES for w in words):
        return None
    body = [(False, w, ("?x",)) for w in words[:-1]]
    if not generic:
        body.append((False, last, ("?x",)))
    pieces = [p.strip() for p in re.split(r"\s*,?\s+and\s+", predicate) if p.strip()]
    rules = []
    for piece in pieces:
        pw = [x for x in piece.split() if x not in ARTICLES]
        negated = bool(negation)
        if pw and pw[0] == "not":
            negated = True
            pw = pw[1:]
        if not pw or any(x in STRICT_REJECT for x in pw):
            return None
        rules.append((list(body), (negated, "_".join(pw), ("?x",))))
    return rules


In [5]:
def parse_horn_sentence(raw):
    s = normalize_sentence(raw)
    if not s:
        return "empty", None
    if s.startswith("if "):
        rules = parse_if_rule(s)
        return ("rules", rules) if rules is not None else ("fail", None)
    universal = parse_universal(s)
    if universal is not None:
        return "rules", universal
    literals = parse_conjunction(s, VarContext())
    if literals is None:
        return "fail", None
    if any(a[:1] == "?" for lit in literals for a in lit[2]):
        return "fail", None
    return "facts", literals


def parse_horn_query(text):
    s = normalize_sentence(text)
    if not s:
        return None
    alternatives = []
    for alternative in re.split(r"\s+or\s+", s):
        literals = parse_conjunction(alternative, VarContext())
        if literals is None:
            return None
        if any(a[:1] == "?" for lit in literals for a in lit[2]):
            return None
        alternatives.append(literals)
    return alternatives


def parse_horn_context(context):
    facts, rules, failures = [], [], []
    total = 0
    for raw in split_sentences(context):
        kind, payload = parse_horn_sentence(raw)
        if kind == "empty":
            continue
        total += 1
        if kind == "fail":
            failures.append(raw)
        elif kind == "facts":
            facts.extend(payload)
        else:
            for body, head in payload:
                rules.append((body, head, raw))
    return facts, rules, failures, total


## 3. Program compilation, stratification and forward chaining

Facts and rule heads retain signed predicates. Negative rule conditions are grounded over the constants in the knowledge base and checked using negation-as-failure after the relevant positive facts have been derived. For a negative query, the engine checks the negative evidence/NAF semantics used by this RuleTaker configuration. Unsupported or unresolved cases are kept separate from a False prediction.


In [6]:
class Rule:
    def __init__(self, uid, pos, neg, head, text):
        self.uid = uid
        self.pos = pos
        self.neg = neg
        self.head = head
        self.text = text
        self.plans = build_plans(pos)


def build_plans(pos):
    plans = []
    for first in range(len(pos)):
        bound = {a for a in pos[first][1] if a[:1] == "?"}
        order = [first]
        remaining = [i for i in range(len(pos)) if i != first]
        while remaining:
            def score(i):
                args = pos[i][1]
                variables = [a for a in args if a[:1] == "?"]
                shared = sum(1 for a in variables if a in bound)
                constants = len(args) - len(variables)
                return (-(shared + constants), len(variables) - shared)
            best = min(remaining, key=score)
            remaining.remove(best)
            order.append(best)
            bound.update(a for a in pos[best][1] if a[:1] == "?")
        plans.append(order)
    return plans


def encode_pred(pred, negated):
    return NEG_PREFIX + pred if negated else pred



def compile_program(fact_literals, rule_triples, owa):
    facts = []
    domain = set()
    for negated, pred, args in fact_literals:
        facts.append((encode_pred(pred, negated), tuple(args)))
        domain.update(a for a in args if not str(a).startswith("?"))

    prepared = []
    uid = 0
    for body, head, text in rule_triples:
        pos, neg = [], []
        for negated, pred, args in body:
            args = tuple(args)
            if negated and owa:
                pos.append((encode_pred(pred, True), args))
            elif negated:
                neg.append((pred, args))
            else:
                pos.append((pred, args))
            domain.update(a for a in args if not str(a).startswith("?"))

        hneg, hpred, hargs = head
        hargs = tuple(hargs)
        domain.update(a for a in hargs if not str(a).startswith("?"))
        head_atom = (encode_pred(hpred, hneg), hargs)

        positively_bound = {
            term
            for _, args in pos
            for term in args
            if isinstance(term, str) and term.startswith("?")
        }
        variables_to_ground = {
            term
            for term in head_atom[1]
            if isinstance(term, str) and term.startswith("?") and term not in positively_bound
        }
        variables_to_ground.update(
            term
            for _, args in neg
            for term in args
            if isinstance(term, str) and term.startswith("?") and term not in positively_bound
        )
        prepared.append((pos, neg, head_atom, text, sorted(variables_to_ground)))

    constants = sorted(domain)
    compiled = []
    for pos, neg, head_atom, text, unsafe in prepared:
        if unsafe and not constants:
            continue
        combinations = itertools.product(constants, repeat=len(unsafe)) if unsafe else [()]
        count = len(constants) ** len(unsafe) if unsafe else 1
        if count > 5000:
            continue

        for values in combinations:
            mapping = dict(zip(unsafe, values))

            def sub(args):
                return tuple(mapping.get(a, a) for a in args)

            p2 = [(p, sub(args)) for p, args in pos]
            n2 = [(p, sub(args)) for p, args in neg]
            h2 = (head_atom[0], sub(head_atom[1]))

            if not p2 and not n2:
                facts.append(h2)
            else:
                compiled.append(Rule(uid, p2, n2, h2, text))
                uid += 1

    return facts, compiled, constants


In [7]:
class FactBase:
    def __init__(self):
        self.rel = defaultdict(set)
        self.idx = defaultdict(dict)
        self.why = {}

    def add(self, pred, args, reason=None):
        rows = self.rel[pred]
        if args in rows:
            return False
        rows.add(args)
        for i, a in enumerate(args):
            bucket = self.idx[(pred, i)]
            if a in bucket:
                bucket[a].append(args)
            else:
                bucket[a] = [args]
        self.why[(pred, args)] = reason
        return True

    def has(self, pred, args):
        return args in self.rel.get(pred, ())

    def size(self):
        return sum(len(v) for v in self.rel.values())


def candidate_rows(fb, pred, pattern, theta):
    best = None
    for i, p in enumerate(pattern):
        value = theta.get(p) if p[:1] == "?" else p
        if value is not None:
            rows = fb.idx.get((pred, i), {}).get(value)
            if rows is None:
                return ()
            if best is None or len(rows) < len(best):
                best = rows
    if best is None:
        return tuple(fb.rel.get(pred, ()))
    return tuple(best)


def match_row(pattern, row, theta):
    if len(pattern) != len(row):
        return None
    out = theta
    copied = False
    for p, v in zip(pattern, row):
        if p[:1] == "?":
            current = out.get(p)
            if current is None:
                if not copied:
                    out = dict(out)
                    copied = True
                out[p] = v
            elif current != v:
                return None
        elif p != v:
            return None
    return out


def substitute(pattern, theta):
    return tuple(theta[p] if p[:1] == "?" else p for p in pattern)


def naf_holds(neg_literals, theta, oracle):
    for pred, pattern in neg_literals:
        if oracle is None:
            return False
        unbound = [p for p in pattern if p[:1] == "?" and p not in theta]
        if not unbound:
            if substitute(pattern, theta) in oracle.rel.get(pred, ()):
                return False
        else:
            for row in candidate_rows(oracle, pred, pattern, theta):
                if match_row(pattern, row, theta) is not None:
                    return False
    return True


def join_rule(rule, plan, delta_rows, fb, oracle):
    results = []
    pos = rule.pos
    support = []

    def extend(k, theta):
        if k == len(plan):
            if naf_holds(rule.neg, theta, oracle):
                results.append((theta, tuple(support)))
            return
        pred, pattern = pos[plan[k]]
        rows = delta_rows if k == 0 else candidate_rows(fb, pred, pattern, theta)
        for row in rows:
            t2 = match_row(pattern, row, theta)
            if t2 is None:
                continue
            support.append((pred, row))
            extend(k + 1, t2)
            support.pop()

    extend(0, {})
    return results


In [8]:
def run_lfp(rules, fb, oracle, goal=None, stats=None):
    for rule in rules:
        if not rule.pos and naf_holds(rule.neg, {}, oracle):
            fb.add(rule.head[0], rule.head[1], (rule.uid, ()))
    delta = {pred: list(rows) for pred, rows in fb.rel.items()}
    if goal is not None and fb.has(*goal):
        return True
    while delta:
        pending = []
        for rule in rules:
            for i, (pred, pattern) in enumerate(rule.pos):
                drows = delta.get(pred)
                if not drows:
                    continue
                for theta, support in join_rule(rule, rule.plans[i], drows, fb, oracle):
                    pending.append((rule, substitute(rule.head[1], theta), support))
                if stats is not None:
                    stats["joins"] += 1
        new_delta = defaultdict(list)
        for rule, args, support in pending:
            if fb.add(rule.head[0], args, (rule.uid, support)):
                new_delta[rule.head[0]].append(args)
        delta = new_delta
        if stats is not None:
            stats["rounds"] += 1
        if goal is not None and fb.has(*goal):
            return True
    return False


In [9]:
def compute_strata(rules):
    preds = set()
    for r in rules:
        preds.add(r.head[0])
        preds.update(p for p, _ in r.pos)
        preds.update(p for p, _ in r.neg)
    level = {p: 0 for p in preds}
    for _ in range(len(preds) + 2):
        changed = False
        for r in rules:
            need = 0
            for p, _ in r.pos:
                need = max(need, level[p])
            for p, _ in r.neg:
                need = max(need, level[p] + 1)
            if need > level[r.head[0]]:
                level[r.head[0]] = need
                changed = True
        if not changed:
            return level
        if max(level.values()) > len(preds):
            return None
    return None


In [10]:
def relevant_rules(rules, goal_preds):
    by_head = defaultdict(list)
    for r in rules:
        by_head[r.head[0]].append(r)
    seen = set(goal_preds)
    queue = deque(goal_preds)
    keep = []
    kept_ids = set()
    while queue:
        p = queue.popleft()
        for r in by_head.get(p, ()):
            if r.uid in kept_ids:
                continue
            kept_ids.add(r.uid)
            keep.append(r)
            for q, _ in list(r.pos) + list(r.neg):
                if q not in seen:
                    seen.add(q)
                    queue.append(q)
    return keep


def evaluate_program(facts, rules, goal=None, stats=None):
    levels = compute_strata(rules)
    if levels is not None:
        fb = FactBase()
        for pred, args in facts:
            fb.add(pred, args, None)
        grouped = defaultdict(list)
        for r in rules:
            grouped[levels[r.head[0]]].append(r)
        for lvl in sorted(grouped):
            final = lvl == max(grouped)
            if run_lfp(grouped[lvl], fb, fb, goal if final else None, stats):
                break
        return fb, "stratified"
    under = None
    over = None
    previous = -1
    for _ in range(100):
        fb_under = FactBase()
        for pred, args in facts:
            fb_under.add(pred, args, None)
        run_lfp(rules, fb_under, over, goal, stats)
        if goal is not None and fb_under.has(*goal):
            return fb_under, "alternating"
        size = fb_under.size()
        if size == previous:
            return fb_under, "alternating"
        previous = size
        fb_over = FactBase()
        for pred, args in facts:
            fb_over.add(pred, args, None)
        run_lfp(rules, fb_over, fb_under, None, stats)
        over = fb_over
        under = fb_under
    return under, "alternating"


## 4. Backward chaining, proof explanation and the Horn knowledge base

In [11]:
class SearchBudgetExceeded(Exception):
    pass


class BackwardChainer:
    def __init__(self, facts, rules, max_depth=BACKWARD_MAX_DEPTH, max_nodes=BACKWARD_MAX_NODES):
        self.facts = defaultdict(set)
        for pred, args in facts:
            self.facts[pred].add(args)
        self.rules = defaultdict(list)
        for r in rules:
            self.rules[r.head[0]].append(r)
        self.max_depth = max_depth
        self.max_nodes = max_nodes
        self.counter = itertools.count()
        self.nodes = 0
        self.proved = set()

    def walk(self, term, theta):
        while term[:1] == "?" and term in theta:
            term = theta[term]
        return term

    def unify_args(self, a, b, theta):
        if len(a) != len(b):
            return None
        out = theta
        copied = False
        for x, y in zip(a, b):
            x = self.walk(x, out)
            y = self.walk(y, out)
            if x == y:
                continue
            if not copied:
                out = dict(out)
                copied = True
            if x[:1] == "?":
                out[x] = y
            elif y[:1] == "?":
                out[y] = x
            else:
                return None
        return out

    def rename(self, rule):
        tag = "#%d" % next(self.counter)

        def r(args):
            return tuple(a + tag if a[:1] == "?" else a for a in args)

        pos = [(p, r(a)) for p, a in rule.pos]
        neg = [(p, r(a)) for p, a in rule.neg]
        return pos, neg, (rule.head[0], r(rule.head[1]))

    def prove(self, pred, args, theta, depth, stack):
        self.nodes += 1
        if self.nodes > self.max_nodes:
            raise SearchBudgetExceeded()
        if depth > self.max_depth:
            return
        key = (pred, tuple(self.walk(a, theta) for a in args))
        if key in self.proved:
            yield theta
            return
        if key in stack:
            return
        ground = all(a[:1] != "?" for a in key[1])
        stack = stack | {key}
        for row in tuple(self.facts.get(pred, ())):
            t2 = self.unify_args(args, row, theta)
            if t2 is not None:
                yield t2
        for rule in self.rules.get(pred, ()):
            pos, neg, head = self.rename(rule)
            t2 = self.unify_args(args, head[1], theta)
            if t2 is None:
                continue
            for t3 in self.prove_body(pos, 0, neg, t2, depth + 1, stack):
                if ground:
                    self.proved.add(key)
                yield t3
                if ground:
                    return

    def prove_body(self, pos, i, neg, theta, depth, stack):
        if i == len(pos):
            for p, a in neg:
                for _ in self.prove(p, a, theta, depth, frozenset()):
                    return
            yield theta
            return
        p, a = pos[i]
        for t2 in self.prove(p, a, theta, depth, stack):
            for t3 in self.prove_body(pos, i + 1, neg, t2, depth, stack):
                yield t3

    def entails(self, pred, args):
        for _ in self.prove(pred, tuple(args), {}, 0, frozenset()):
            return True
        return False


def explain_fact(fb, rules_by_uid, pred, args, depth=0, seen=None, lines=None, limit=60):
    if lines is None:
        lines = []
    if seen is None:
        seen = set()
    if len(lines) >= limit:
        return lines
    shown = "%s(%s)" % (pred, ", ".join(args))
    reason = fb.why.get((pred, args))
    indent = "  " * depth
    if reason is None:
        lines.append(indent + shown + "  [given fact]")
        return lines
    uid, support = reason
    rule = rules_by_uid.get(uid)
    lines.append(indent + shown + "  [by rule: " + (rule.text if rule else "?") + "]")
    if (pred, args) in seen:
        return lines
    seen.add((pred, args))
    for sp, sa in support:
        explain_fact(fb, rules_by_uid, sp, sa, depth + 1, seen, lines, limit)
    for np_, na in (rule.neg if rule else ()):
        lines.append(indent + "  not %s(...)  [negation as failure]" % np_)
    return lines


In [12]:
class HornKnowledgeBase:
    def __init__(self, context, negation_mode="naf"):
        facts, rule_triples, failures, total = parse_horn_context(context)
        self.parse_total = total
        self.parse_failures = failures
        self.raw_facts = facts
        self.raw_rules = rule_triples
        self.negation_mode = negation_mode
        self.owa = negation_mode == "explicit"
        self.facts, self.rules, self.constants = compile_program(facts, rule_triples, self.owa)
        self.rules_by_uid = {rule.uid: rule for rule in self.rules}

    def parse_quality(self):
        return 1.0 - len(self.parse_failures) / self.parse_total if self.parse_total else 0.0

    def literal_holds(self, literal, engine="forward", stats=None):
        negated, pred, args = literal
        args = tuple(args)
        negative_pred = encode_pred(pred, True)

        if engine == "backward":
            try:
                chainer = BackwardChainer(self.facts, self.rules)
                positive = chainer.entails(pred, args)
                if not negated:
                    return positive
                if self.owa:
                    return chainer.entails(negative_pred, args)
                return not positive
            except SearchBudgetExceeded:
                engine = "forward"

        goals = [negative_pred] if negated and self.owa else [pred]
        rules = relevant_rules(self.rules, goals)
        fact_base, mode = evaluate_program(self.facts, rules, None, stats)
        positive = fact_base.has(pred, args)
        if not negated:
            return positive, fact_base, mode
        if self.owa:
            holds = fact_base.has(negative_pred, args)
        else:
            holds = not positive
        return holds, fact_base, mode

    def answer(self, alternatives, engine="forward", want_trace=False):
        stats = {"joins": 0, "rounds": 0}
        trace = []
        for conjunction in alternatives:
            accepted = True
            for literal in conjunction:
                outcome = self.literal_holds(literal, engine, stats)
                if isinstance(outcome, bool):
                    holds, fact_base, mode = outcome, None, "backward"
                else:
                    holds, fact_base, mode = outcome
                if want_trace and fact_base is not None:
                    negated, pred, args = literal
                    if holds and not negated:
                        trace.extend(explain_fact(fact_base, self.rules_by_uid, pred, tuple(args)))
                    elif holds and negated:
                        trace.append("not %s(%s) holds by negation as failure" % (pred, ", ".join(args)))
                    else:
                        trace.append("%s%s(%s) is not established" % ("not " if negated else "", pred, ", ".join(args)))
                if not holds:
                    accepted = False
                    break
            if accepted:
                return True, trace, stats
        return False, trace, stats


## 5. First-order machinery: unification, CNF conversion and resolution prover

In [13]:
def walk_term(t, s):
    while isinstance(t, str) and t[:1] == "?" and t in s:
        t = s[t]
    return t


def occurs_in(v, t, s):
    t = walk_term(t, s)
    if t == v:
        return True
    if isinstance(t, tuple):
        return any(occurs_in(v, x, s) for x in t[1:])
    return False


def unify_term(a, b, s):
    a = walk_term(a, s)
    b = walk_term(b, s)
    if a == b:
        return s
    if is_variable(a):
        if occurs_in(a, b, s):
            return None
        s2 = dict(s)
        s2[a] = b
        return s2
    if is_variable(b):
        if occurs_in(b, a, s):
            return None
        s2 = dict(s)
        s2[b] = a
        return s2
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            s = unify_term(x, y, s)
            if s is None:
                return None
        return s
    return None


def unify_args(a, b, s):
    if len(a) != len(b):
        return None
    for x, y in zip(a, b):
        s = unify_term(x, y, s)
        if s is None:
            return None
    return s


def apply_term(t, s):
    t = walk_term(t, s)
    if isinstance(t, tuple):
        return (t[0],) + tuple(apply_term(x, s) for x in t[1:])
    return t


def apply_literal(lit, s):
    return (lit[0], lit[1], tuple(apply_term(a, s) for a in lit[2]))


def rename_term(t, tag):
    if isinstance(t, tuple):
        return (t[0],) + tuple(rename_term(x, tag) for x in t[1:])
    if t[:1] == "?":
        return t + "_" + tag
    return t


def rename_clause(clause, tag):
    return frozenset((s, p, tuple(rename_term(a, tag) for a in args)) for s, p, args in clause)


In [14]:
def skeleton(t):
    if isinstance(t, tuple):
        return "(" + t[0] + " " + " ".join(skeleton(x) for x in t[1:]) + ")"
    return "?" if t[:1] == "?" else t


def canon_term(t, mapping):
    if isinstance(t, tuple):
        return (t[0],) + tuple(canon_term(x, mapping) for x in t[1:])
    if t[:1] == "?":
        if t not in mapping:
            mapping[t] = "?%d" % len(mapping)
        return mapping[t]
    return t


def canonical_clause(clause):
    ordered = sorted(clause, key=lambda l: (l[1], l[0], " ".join(skeleton(a) for a in l[2])))
    mapping = {}
    out = []
    for sign, pred, args in ordered:
        out.append((sign, pred, tuple(canon_term(a, mapping) for a in args)))
    return frozenset(out)


def term_weight(t):
    if isinstance(t, tuple):
        return 1 + sum(term_weight(x) for x in t[1:])
    return 1


def clause_weight(clause):
    return sum(1 + sum(term_weight(a) for a in args) for _, _, args in clause)


def is_tautology(clause):
    for sign, pred, args in clause:
        if (not sign, pred, args) in clause:
            return True
    return False


In [15]:
def match_one_way(pattern, target, s):
    if isinstance(pattern, tuple):
        if not isinstance(target, tuple) or pattern[0] != target[0] or len(pattern) != len(target):
            return None
        for x, y in zip(pattern[1:], target[1:]):
            s = match_one_way(x, y, s)
            if s is None:
                return None
        return s
    if pattern[:1] == "?":
        if pattern in s:
            return s if s[pattern] == target else None
        s2 = dict(s)
        s2[pattern] = target
        return s2
    return s if pattern == target else None


def factor_clause(clause):
    lits = list(clause)
    out = []
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            a, b = lits[i], lits[j]
            if a[0] == b[0] and a[1] == b[1] and len(a[2]) == len(b[2]):
                s = unify_args(a[2], b[2], {})
                if s is not None:
                    out.append(frozenset(apply_literal(l, s) for l in lits))
    return out


In [16]:
class ResolutionProver:
    def __init__(self, max_clauses=RESOLUTION_MAX_CLAUSES, max_seconds=RESOLUTION_MAX_SECONDS, max_weight=RESOLUTION_MAX_WEIGHT):
        self.max_clauses = max_clauses
        self.max_seconds = max_seconds
        self.max_weight = max_weight
        self.reset()

    def reset(self):
        self.clauses = []
        self.parents = []
        self.index = defaultdict(list)
        self.units = defaultdict(list)
        self.seen = set()
        self.queue = []
        self.tick = itertools.count()
        self.generated = 0
        self.empty_id = None
        self.truncated = False
        self.budget_exceeded = False

    def subsumed_by_unit(self, clause):
        for sign, pred, args in clause:
            for u_args in self.units.get((sign, pred), ()):
                if len(u_args) == len(args):
                    s = {}
                    for p, t in zip(u_args, args):
                        s = match_one_way(p, t, s)
                        if s is None:
                            break
                    if s is not None:
                        return True
        return False

    def register(self, clause, parent):
        if is_tautology(clause):
            return None
        clause = canonical_clause(clause)
        if clause in self.seen:
            return None
        if clause and clause_weight(clause) > self.max_weight:
            self.truncated = True
            return None
        if len(self.clauses) >= self.max_clauses:
            self.truncated = True
            return None
        if clause and self.subsumed_by_unit(clause):
            return None
        self.seen.add(clause)
        cid = len(self.clauses)
        self.clauses.append(clause)
        self.parents.append(parent)
        if not clause:
            self.empty_id = cid
        return cid

    def activate(self, cid):
        clause = self.clauses[cid]
        for sign, pred, args in clause:
            self.index[(sign, pred)].append(cid)
        if len(clause) == 1:
            sign, pred, args = next(iter(clause))
            self.units[(sign, pred)].append(args)

    def push(self, cid):
        heapq.heappush(self.queue, (clause_weight(self.clauses[cid]), next(self.tick), cid))

    def resolvents(self, gid):
        given = self.clauses[gid]
        out = []
        for lit in given:
            sign, pred, args = lit
            for oid in list(self.index.get((not sign, pred), ())):
                other = rename_clause(self.clauses[oid], "r%d" % self.generated)
                for lit2 in other:
                    if lit2[0] == sign or lit2[1] != pred:
                        continue
                    s = unify_args(args, lit2[2], {})
                    if s is None:
                        continue
                    rest = [apply_literal(l, s) for l in given if l != lit]
                    rest += [apply_literal(l, s) for l in other if l != lit2]
                    out.append((frozenset(rest), (gid, oid)))
                    self.generated += 1
        return out

    def prove(self, kb_clauses, goal_clauses):
        self.reset()
        deadline = time.perf_counter() + self.max_seconds
        for c in kb_clauses:
            cid = self.register(c, None)
            if cid is not None:
                self.activate(cid)
        for c in goal_clauses:
            cid = self.register(c, None)
            if cid is not None:
                self.push(cid)
            if self.empty_id is not None:
                return True
        while self.queue:
            if len(self.clauses) >= self.max_clauses or time.perf_counter() > deadline:
                self.budget_exceeded = True
                return None
            _, _, gid = heapq.heappop(self.queue)
            self.activate(gid)
            candidates = self.resolvents(gid)
            for f in factor_clause(self.clauses[gid]):
                candidates.append((f, (gid,)))
            for clause, parent in candidates:
                cid = self.register(clause, parent)
                if cid is None:
                    continue
                if self.empty_id is not None:
                    return True
                self.push(cid)
        return None if self.truncated else False

    def proof_lines(self):
        if self.empty_id is None:
            return []
        order = []
        seen = set()

        def visit(cid):
            if cid in seen:
                return
            seen.add(cid)
            parent = self.parents[cid]
            if parent:
                for p in parent:
                    visit(p)
            order.append(cid)

        visit(self.empty_id)
        names = {cid: i + 1 for i, cid in enumerate(order)}
        lines = []
        for cid in order:
            parent = self.parents[cid]
            text = format_clause(self.clauses[cid])
            if parent:
                lines.append("%d. %s   [resolve %s]" % (names[cid], text, ", ".join(str(names[p]) for p in parent)))
            else:
                lines.append("%d. %s   [premise / negated goal]" % (names[cid], text))
        return lines


In [17]:
def format_term(t):
    if isinstance(t, tuple):
        return t[0] + "(" + ", ".join(format_term(x) for x in t[1:]) + ")"
    return t


def format_clause(clause):
    if not clause:
        return "[] (contradiction)"
    parts = []
    for sign, pred, args in sorted(clause, key=lambda l: (l[1], l[0])):
        body = pred + "(" + ", ".join(format_term(a) for a in args) + ")"
        parts.append(body if sign else "~" + body)
    return " | ".join(parts)


In [18]:
def eliminate_implications(f):
    op = f[0]
    if op in ("atom", "true", "false"):
        return f
    if op == "not":
        return ("not", eliminate_implications(f[1]))
    if op in ("and", "or"):
        return (op,) + tuple(eliminate_implications(x) for x in f[1:])
    if op == "imp":
        return ("or", ("not", eliminate_implications(f[1])), eliminate_implications(f[2]))
    if op == "iff":
        a = eliminate_implications(f[1])
        b = eliminate_implications(f[2])
        return ("and", ("or", ("not", a), b), ("or", ("not", b), a))
    return (op, f[1], eliminate_implications(f[2]))


def push_negation(f, negate=False):
    op = f[0]
    if op == "atom":
        return ("not", f) if negate else f
    if op == "true":
        return ("false",) if negate else f
    if op == "false":
        return ("true",) if negate else f
    if op == "not":
        return push_negation(f[1], not negate)
    if op in ("and", "or"):
        flipped = {"and": "or", "or": "and"}[op] if negate else op
        return (flipped,) + tuple(push_negation(x, negate) for x in f[1:])
    flipped = {"all": "ex", "ex": "all"}[op] if negate else op
    return (flipped, f[1], push_negation(f[2], negate))


In [19]:
def skolemize(f, universals, mapping, counters):
    op = f[0]
    if op == "atom":
        return ("atom", f[1], tuple(mapping.get(a, a) if isinstance(a, str) else a for a in f[2]))
    if op == "not":
        return ("not", skolemize(f[1], universals, mapping, counters))
    if op in ("and", "or"):
        return (op,) + tuple(skolemize(x, universals, mapping, counters) for x in f[1:])
    if op in ("true", "false"):
        return f
    if op == "all":
        fresh = "?u%d" % next(counters["var"])
        return skolemize(f[2], universals + [fresh], dict(mapping, **{f[1]: fresh}), counters)
    name = "sk%d" % next(counters["sk"])
    term = (name,) + tuple(universals) if universals else name
    return skolemize(f[2], universals, dict(mapping, **{f[1]: term}), counters)


def distribute(f):
    op = f[0]
    if op == "and":
        out = []
        for x in f[1:]:
            out.extend(distribute(x))
        return out
    if op == "or":
        result = [frozenset()]
        for x in f[1:]:
            part = distribute(x)
            result = [a | b for a in result for b in part]
            if len(result) > CNF_MAX_CLAUSES:
                raise OverflowError("cnf too large")
        return result
    if op == "atom":
        return [frozenset([(True, f[1], f[2])])]
    if op == "not":
        a = f[1]
        return [frozenset([(False, a[1], a[2])])]
    if op == "true":
        return []
    return [frozenset()]


def to_cnf(formula, counters):
    f = eliminate_implications(formula)
    f = push_negation(f)
    f = skolemize(f, [], {}, counters)
    clauses = distribute(f)
    return [c for c in clauses if not is_tautology(c)]


def free_variables(f, bound=frozenset()):
    op = f[0]
    if op == "atom":
        return {a for a in f[2] if isinstance(a, str) and a[:1] == "?" and a not in bound}
    if op in ("true", "false"):
        return set()
    if op == "not":
        return free_variables(f[1], bound)
    if op in ("and", "or", "imp", "iff"):
        out = set()
        for x in f[1:]:
            out |= free_variables(x, bound)
        return out
    return free_variables(f[2], bound | {f[1]})


def close_formula(f):
    for v in sorted(free_variables(f)):
        f = ("all", v, f)
    return f


def connected_clauses(kb_clauses, goal_clauses):
    goal_preds = {p for c in goal_clauses for _, p, _ in c}
    reached = set(goal_preds)
    keep = [False] * len(kb_clauses)
    changed = True
    while changed:
        changed = False
        for i, c in enumerate(kb_clauses):
            if keep[i]:
                continue
            preds = {p for _, p, _ in c}
            if preds & reached:
                keep[i] = True
                reached |= preds
                changed = True
    return [c for i, c in enumerate(kb_clauses) if keep[i]]


## 6. Natural language to first-order logic (FOLIO style)

In [20]:
QUANTIFIERS_ALL = {"all", "every", "each", "any"}
RELATIVE_WORDS = {"that", "who", "which", "whom"}
NON_VERB_S = {"this", "his", "its", "as", "us", "yes", "always", "sometimes", "less", "unless", "was", "is", "has", "does", "perhaps", "across", "various", "famous", "previous", "serious", "dangerous", "his", "chess", "gas", "bus", "business", "class", "glass", "grass", "lens", "series"}
PRONOUN_SUBJECTS = {"he", "she", "they", "it", "this", "that"}
AUX_START = COPULAS | DO_AUX | MODAL_VERBS | HAVE_FORMS


class FolState:
    def __init__(self):
        self.counters = {"var": itertools.count(1), "sk": itertools.count(1)}
        self.var_counter = itertools.count(1)
        self.last_subject = None
        self.in_condition = False
        self.failed = []

    def fresh_var(self):
        return "?v%d" % next(self.var_counter)


def tokenize_fol(sentence):
    s = normalize_sentence(sentence)
    return re.findall(r"[^\s,;]+|,|;", s)


def verb_like(tokens, i):
    t = tokens[i]
    if not t.isalpha() or len(t) < 3 or t in NON_VERB_S or t in ARTICLES:
        return False
    if not t.endswith("s"):
        return False
    if i > 0 and (tokens[i - 1] in QUANTIFIERS_ALL or tokens[i - 1] in ARTICLES or tokens[i - 1] in ("some", "no")):
        return False
    return i >= 1 or len(tokens) >= 2


def predicate_start(tokens):
    for i, t in enumerate(tokens):
        if t in AUX_START:
            return i
        if verb_like(tokens, i):
            return i
    return None


def has_predicate(tokens):
    p = predicate_start(tokens)
    return p is not None and p >= 1


def find_seq(tokens, seq, start=0):
    n = len(seq)
    for i in range(start, len(tokens) - n + 1):
        if tokens[i:i + n] == seq:
            return i
    return -1


def strip_commas(tokens):
    return [t for t in tokens if t not in (",", ";")]


def split_connective(tokens, word):
    segments = []
    current = []
    for t in tokens:
        if t == word and current and has_predicate(strip_commas(current)):
            segments.append(current)
            current = []
        else:
            current.append(t)
    segments.append(current)
    segments = [strip_commas(s) for s in segments]
    segments = [s for s in segments if s]
    return segments if len(segments) > 1 else None


def vp_prefix(tokens):
    if tokens and tokens[0] in COPULAS:
        return [tokens[0]]
    i = 0
    while i < len(tokens) and (tokens[i] in AUX_START or tokens[i] == "not"):
        i += 1
    return tokens[:i + 1]


def piece_is_standalone_vp(piece):
    if not piece:
        return False
    if piece[0] in AUX_START or piece[0] == "not":
        return True
    return len(piece) >= 2 and verb_like(piece, 0) and piece[0] not in ARTICLES


def constant_from_np(tokens, state=None):
    words = [t for t in tokens if t not in ARTICLES and t not in (",", ";")]
    if not words:
        return None
    if len(words) == 1 and words[0] in PRONOUN_SUBJECTS and state is not None and state.last_subject is not None:
        return state.last_subject
    return "_".join(words)


In [21]:
def noun_predicate(tokens, var):
    words = [t for t in tokens if t not in ARTICLES and t not in (",", ";")]
    if not words:
        return ("true",)
    last = singular(words[-1])
    if len(words) == 1 and last in GENERIC_NOUNS:
        return ("true",)
    if last in GENERIC_NOUNS and len(words) > 1:
        words = words[:-1]
        return ("atom", "_".join(words), (var,))
    return ("atom", "_".join(words[:-1] + [last]), (var,))


def combine(op, parts):
    parts = [p for p in parts if p is not None]
    if not parts:
        return None
    if len(parts) == 1:
        return parts[0]
    return (op,) + tuple(parts)


def copular_atom(tokens, subject, state, negated, plural=False):
    words = list(tokens)
    extra = None
    rel = next((i for i, t in enumerate(words) if t in RELATIVE_WORDS), None)
    if rel is not None:
        extra = prop_expr(words[rel + 1:], subject, state)
        words = words[:rel]
    if not words:
        return None
    prep = next((i for i, t in enumerate(words) if t in PREPOSITIONS and i > 0), None)
    if prep is not None:
        obj = constant_from_np(words[prep + 1:], state)
        if obj is None:
            return None
        pred = "_".join(w for w in words[:prep + 1] if w not in ARTICLES)
        atom = ("atom", pred, (subject, obj))
    else:
        content = [w for w in words if w not in ARTICLES]
        if not content:
            return None
        if words[0] in ("a", "an") or plural or content[-1] in IRREGULAR_SINGULAR:
            content[-1] = singular(content[-1])
        atom = ("atom", "_".join(content), (subject,))
    f = atom if extra is None else ("and", atom, extra)
    return ("not", f) if negated else f


In [22]:
def prop_atom(tokens, subject, state):
    toks = list(tokens)
    if not toks:
        return None
    modal = None
    if toks[0] in MODAL_VERBS:
        modal = toks[0]
        toks = toks[1:]
    if toks and toks[0] in DO_AUX:
        toks = toks[1:]
    copular = False
    plural = False
    if toks and toks[0] in COPULAS:
        copular = True
        plural = toks[0] in ("are", "were")
        toks = toks[1:]
    negated = False
    if toks and toks[0] == "not":
        negated = True
        toks = toks[1:]
    if not toks:
        return None
    if copular:
        return copular_atom(toks, subject, state, negated, plural)
    verb = singular(toks[0])
    pred = (modal + "_" + verb) if modal else verb
    rest = toks[1:]
    if not rest:
        atom = ("atom", pred, (subject,))
    else:
        rel = next((i for i, t in enumerate(rest) if t in RELATIVE_WORDS), None)
        extra = None
        if rel is not None:
            extra = prop_expr(rest[rel + 1:], constant_from_np(rest[:rel], state) or subject, state)
            rest = rest[:rel]
        obj = constant_from_np(rest, state)
        if obj is None:
            return None
        atom = ("atom", pred, (subject, obj))
        if extra is not None:
            atom = ("and", atom, extra)
    return ("not", atom) if negated else atom


def prop_expr(tokens, subject, state):
    tokens = strip_commas(tokens)
    if not tokens:
        return None
    if tokens[0] == "both":
        tokens = tokens[1:]
    for word, op in (("or", "or"), ("and", "and")):
        if word in tokens[1:]:
            pieces = []
            current = []
            for t in tokens:
                if t == word and current:
                    pieces.append(current)
                    current = []
                else:
                    current.append(t)
            pieces.append(current)
            pieces = [p for p in pieces if p]
            if len(pieces) > 1:
                parts = []
                prefix = None
                for piece in pieces:
                    if piece_is_standalone_vp(piece):
                        full = piece
                    elif prefix is not None:
                        full = prefix + piece
                    else:
                        full = piece
                    parts.append(prop_atom(full, subject, state))
                    prefix = vp_prefix(full)
                if any(p is None for p in parts):
                    return None
                return combine(op, parts)
    return prop_atom(tokens, subject, state)


In [23]:
def parse_quantified(tokens, state, kind):
    var = state.fresh_var()
    tokens = strip_commas(tokens)
    rel = next((i for i, t in enumerate(tokens) if t in RELATIVE_WORDS), None)
    if rel is not None:
        noun_tokens = tokens[:rel]
        j = rel + 1
        if j < len(tokens) and tokens[j] in AUX_START:
            j += 1
        m = next((i for i in range(j + 1, len(tokens)) if tokens[i] in AUX_START), None)
        if m is None:
            m = next((i for i in range(j + 1, len(tokens)) if verb_like(tokens, i)), None)
        if m is None:
            return None
        restrictor_vp = tokens[rel + 1:m]
        main_vp = tokens[m:]
        restrictor = combine("and", [noun_predicate(noun_tokens, var), prop_expr(restrictor_vp, var, state)])
    else:
        m = next((i for i in range(1, len(tokens)) if tokens[i] in AUX_START), None)
        if m is None:
            m = next((i for i in range(1, len(tokens)) if verb_like(tokens, i)), None)
        if m is None:
            noun_tokens, main_vp = tokens[:1], tokens[1:]
        else:
            noun_tokens, main_vp = tokens[:m], tokens[m:]
        restrictor = noun_predicate(noun_tokens, var)
    if restrictor is None or not main_vp:
        return None
    body = prop_expr(main_vp, var, state)
    if body is None:
        return None
    state.last_subject = var
    if kind == "all":
        return ("all", var, ("imp", restrictor, body))
    if kind == "none":
        return ("all", var, ("imp", restrictor, ("not", body)))
    if kind == "some":
        return ("ex", var, ("and", restrictor, body))
    return ("ex", var, ("and", restrictor, ("not", body)))


def parse_clause(tokens, state):
    tokens = strip_commas(tokens)
    if not tokens:
        return None
    if tokens[:2] == ["not", "all"]:
        return parse_quantified(tokens[2:], state, "notall")
    first = tokens[0]
    if first in QUANTIFIERS_ALL:
        return parse_quantified(tokens[1:], state, "all")
    if first in ("everyone", "everybody", "everything") and len(tokens) > 1 and tokens[1] in RELATIVE_WORDS:
        return parse_quantified(tokens[1:], state, "all")
    if first in ("someone", "somebody", "something") and len(tokens) > 1 and tokens[1] in RELATIVE_WORDS:
        return parse_quantified(tokens[1:], state, "some")
    if first in ("everyone", "everybody", "everything"):
        var = state.fresh_var()
        body = prop_expr(tokens[1:], var, state)
        return ("all", var, body) if body is not None else None
    if first in ("nobody", "noone", "nothing") or tokens[:2] == ["no", "one"]:
        var = state.fresh_var()
        vp = tokens[2:] if first == "no" else tokens[1:]
        body = prop_expr(vp, var, state)
        return ("all", var, ("not", body)) if body is not None else None
    if first == "no":
        return parse_quantified(tokens[1:], state, "none")
    if first == "some" or tokens[:3] == ["at", "least", "one"]:
        return parse_quantified(tokens[1:] if first == "some" else tokens[3:], state, "some")
    if tokens[:3] in (["there", "is", "a"], ["there", "is", "an"], ["there", "are", "some"], ["there", "exists", "a"]):
        return parse_quantified(tokens[3:], state, "some")
    if first in ("someone", "somebody", "something"):
        var = state.fresh_var()
        body = prop_expr(tokens[1:], var, state)
        if body is None:
            return None
        if state.in_condition:
            state.last_subject = var
            return body
        return ("ex", var, body)
    if first in ("a", "an") and len(tokens) > 2:
        if True:
            p = predicate_start(tokens[1:])
            if p is not None and p >= 1:
                var = state.fresh_var()
                noun_tokens = tokens[1:1 + p]
                body = prop_expr(tokens[1 + p:], var, state)
                if body is None:
                    return None
                state.last_subject = var
                if state.in_condition:
                    return combine("and", [noun_predicate(noun_tokens, var), body])
                return ("all", var, ("imp", noun_predicate(noun_tokens, var), body))
    p = predicate_start(tokens)
    if p is None or p == 0:
        return None
    subject_tokens = tokens[:p]
    vp = tokens[p:]
    groups = [g for g in split_on_word(subject_tokens, "and") if g]
    subjects = []
    for g in groups:
        c = constant_from_np(g, state)
        if c is None:
            return None
        subjects.append(c)
    parts = [prop_expr(vp, s, state) for s in subjects]
    if any(x is None for x in parts):
        return None
    if subjects and not (len(subjects) == 1 and subjects[0] == state.last_subject):
        state.last_subject = subjects[0]
    return combine("and", parts)


In [24]:
def split_on_word(tokens, word):
    groups = [[]]
    for t in tokens:
        if t == word:
            groups.append([])
        else:
            groups[-1].append(t)
    return groups


def parse_formula(tokens, state, depth=0):
    if depth > 12 or not tokens:
        return None
    tokens = list(tokens)
    while tokens and tokens[0] in (",", ";"):
        tokens = tokens[1:]
    if tokens[:5] == ["it", "is", "not", "the", "case"] and len(tokens) > 6 and tokens[5] == "that":
        inner = parse_formula(tokens[6:], state, depth + 1)
        return ("not", inner) if inner is not None else None
    if tokens[:4] in (["it", "is", "not", "true"], ["it", "is", "false", "that"]) and len(tokens) > 4:
        rest = tokens[4:]
        if rest and rest[0] == "that":
            rest = rest[1:]
        inner = parse_formula(rest, state, depth + 1)
        return ("not", inner) if inner is not None else None
    for seq in (["if", "and", "only", "if"], ["iff"]):
        i = find_seq(tokens, seq, 1)
        if i > 0:
            a = parse_formula(tokens[:i], state, depth + 1)
            b = parse_formula(tokens[i + len(seq):], state, depth + 1)
            return ("iff", a, b) if a is not None and b is not None else None
    if tokens[0] == "if":
        j = find_seq(tokens, ["then"])
        k = tokens.index(",") if "," in tokens else -1
        cut = j if j > 0 else k
        if cut < 0:
            return None
        saved = state.in_condition
        state.in_condition = True
        a = parse_formula(tokens[1:cut], state, depth + 1)
        state.in_condition = saved
        b = parse_formula(tokens[cut + 1:], state, depth + 1)
        if a is None or b is None:
            return None
        return ("imp", a, b)
    i = find_seq(tokens, ["only", "if"], 1)
    if i > 0:
        a = parse_formula(tokens[:i], state, depth + 1)
        b = parse_formula(tokens[i + 2:], state, depth + 1)
        return ("imp", a, b) if a is not None and b is not None else None
    if "if" in tokens[1:]:
        i = tokens.index("if", 1)
        saved = state.in_condition
        state.in_condition = True
        b = parse_formula(tokens[i + 1:], state, depth + 1)
        state.in_condition = saved
        a = parse_formula(tokens[:i], state, depth + 1)
        return ("imp", b, a) if a is not None and b is not None else None
    if "unless" in tokens[1:]:
        i = tokens.index("unless", 1)
        a = parse_formula(tokens[:i], state, depth + 1)
        b = parse_formula(tokens[i + 1:], state, depth + 1)
        return ("or", a, b) if a is not None and b is not None else None
    if tokens[0] == "neither" and "nor" in tokens:
        i = tokens.index("nor")
        a = parse_formula(tokens[1:i], state, depth + 1)
        b = parse_formula(tokens[i + 1:], state, depth + 1)
        if a is not None and b is not None:
            return ("and", ("not", a), ("not", b))
        return None
    if "either" in tokens and "or" in tokens:
        tokens = [t for t in tokens if t != "either"]
    if "both" in tokens and "and" in tokens:
        tokens = [t for t in tokens if t != "both"]
    for word in ("but", "or", "and"):
        pieces = split_connective(tokens, word)
        if pieces:
            parts = []
            previous = None
            for piece in pieces:
                full = piece
                if previous is not None and not has_predicate(piece):
                    ps = predicate_start(previous)
                    if ps is not None and ps >= 1:
                        if piece_is_standalone_vp(piece):
                            full = previous[:ps] + piece
                        else:
                            full = previous[:ps] + vp_prefix(previous[ps:]) + piece
                parts.append(parse_formula(full, state, depth + 1))
                previous = full
            if any(p is None for p in parts):
                return None
            return combine("and" if word in ("and", "but") else "or", parts)
    return parse_clause(tokens, state)


In [25]:
def parse_fol_sentence(sentence, state):
    tokens = tokenize_fol(sentence)
    state.in_condition = False
    f = parse_formula(tokens, state)
    if f is None:
        return None
    return close_formula(f)


## 7. Causal reasoning for CounterBench

This module parses signed causal mechanisms, observations, Bernoulli exogenous variables, and interventions. It performs exact abduction over the modeled noise variables, applies interventions before forward simulation, and estimates the counterfactual outcome. Unsupported or cyclic causal structures should be reported as unresolved rather than silently interpreted as graph reachability.


In [26]:
CAUSAL_MARKERS = ("direct effect", "causal relationship", "intervene", "counterfactual", "hypothetical world")
FOLIO_MARKERS = {"some", "no", "either", "neither", "nor", "or", "who", "which", "every", "each", "any", "unless", "iff", "only", "both", "most", "whom", "whose", "least"}


def split_name_list(text):
    parts = re.split(r",\s*(?:and\s+)?|\s+and\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def clean_variable_name(name):
    name = name.lower().strip()
    name = re.sub(r"^(if|when|whether|that|the|a|an)\s+", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip(" .,")


In [27]:
class CausalGraph:
    def __init__(self):
        self.children = defaultdict(set)
        self.nodes = set()

    def add_edge(self, a, b):
        self.children[a].add(b)
        self.nodes.update((a, b))

    def reachable(self, source, target):
        if source == target:
            return True
        queue = deque([source])
        seen = {source}
        while queue:
            u = queue.popleft()
            for v in self.children.get(u, ()):
                if v == target:
                    return True
                if v not in seen:
                    seen.add(v)
                    queue.append(v)
        return False

    def path(self, source, target):
        queue = deque([source])
        parent = {source: None}
        while queue:
            u = queue.popleft()
            if u == target:
                out = []
                while u is not None:
                    out.append(u)
                    u = parent[u]
                return out[::-1]
            for v in self.children.get(u, ()):
                if v not in parent:
                    parent[v] = u
                    queue.append(v)
        return None

    @staticmethod
    def from_text(text):
        graph = CausalGraph()
        lowered = text.lower()
        patterns = [
            r"([^.]+?)\s+(?:has|have)\s+(?:a\s+)?direct\s+effects?\s+on\s+([^.]+?)(?:\.|$)",
            r"([^.]+?)\s+(?:directly\s+)?(?:causes?|affects?|influences?)\s+([^.]+?)(?:\.|$)",
        ]
        for pattern in patterns:
            for m in re.finditer(pattern, lowered):
                left = [clean_variable_name(x) for x in split_name_list(m.group(1))]
                right = [clean_variable_name(x) for x in split_name_list(m.group(2))]
                for a in left:
                    for b in right:
                        if a and b and a != b:
                            graph.add_edge(a, b)
            if graph.nodes:
                break
        return graph if graph.nodes else None


In [28]:
def causal_answer(text, question):
    graph = CausalGraph.from_text(text)
    if graph is None:
        return None
    q = question.lower()
    found = []
    for node in sorted(graph.nodes, key=len, reverse=True):
        for m in re.finditer(re.escape(node), q):
            found.append((m.start(), m.end(), node))
    found.sort()
    ordered = []
    last_end = -1
    for start, end, node in found:
        if start >= last_end:
            ordered.append(node)
            last_end = end
    if len(ordered) < 2:
        return None
    source, target = ordered[0], ordered[-1]
    if source == target:
        return None
    route = graph.path(source, target)
    return (route is not None), route


In [29]:
def _causal_name(text):
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(text).strip()).strip("_").lower()


def _folio_expand_xor(expression):
    expression = _strip_excess_trailing_parentheses(str(expression).strip())
    while _parenthesis_balance(expression) > 0:
        expression += ")"
    output = []
    i = 0
    while i < len(expression):
        if expression[i] == "(":
            depth = 1
            j = i + 1
            while j < len(expression) and depth:
                if expression[j] == "(":
                    depth += 1
                elif expression[j] == ")":
                    depth -= 1
                j += 1
            if depth == 0:
                output.append("(" + _folio_expand_xor(expression[i + 1:j - 1]) + ")")
                i = j
                continue
        output.append(expression[i])
        i += 1
    expression = "".join(output)
    depth = 0
    for i, char in enumerate(expression):
        if char == "(":
            depth += 1
        elif char == ")":
            depth -= 1
        elif char == "⊕" and depth == 0:
            left = expression[:i].strip()
            right = expression[i + 1:].strip()
            return f"(({left} | {right}) & -({left} & {right}))"
    return expression



def _parenthesis_balance(text):
    return text.count("(") - text.count(")")


def _matching_close_parenthesis(text, start):
    depth = 0
    for index in range(start, len(text)):
        if text[index] == "(":
            depth += 1
        elif text[index] == ")":
            depth -= 1
            if depth == 0:
                return index
    return None


def _strip_excess_trailing_parentheses(text):
    text = text.strip()
    while _parenthesis_balance(text) < 0 and text.endswith(")"):
        text = text[:-1].rstrip()
    return text


def _normalize_folio_formula(formula):
    formula = _folio_expand_xor(str(formula).strip())
    formula = re.sub(r"∀\s*([A-Za-z][A-Za-z0-9_]*)\s*", r"all \1.", formula)
    formula = re.sub(r"∃\s*([A-Za-z][A-Za-z0-9_]*)\s*", r"exists \1.", formula)
    for source, target in {"¬": "-", "∧": "&", "∨": "|", "→": "->", "↔": "<->"}.items():
        formula = formula.replace(source, target)
    formula = re.sub(r"\s+", " ", formula).strip()

    quantifier = re.match(r"^(all|exists)\s+([A-Za-z][A-Za-z0-9_]*)\.\s*", formula)
    if quantifier:
        prefix = quantifier.group(0)
        rest = formula[quantifier.end():]
        if rest.startswith("("):
            close = _matching_close_parenthesis(rest, 0)
            if close is not None:
                tail = rest[close + 1:]
                operator = re.match(r"\s*(<->|->)\s*", tail)
                if operator and _parenthesis_balance(tail) < 0:
                    left = rest[:close + 1]
                    right = _strip_excess_trailing_parentheses(tail[operator.end():])
                    formula = prefix + "(" + left + " " + operator.group(1) + " (" + right + "))"

    formula = _strip_excess_trailing_parentheses(formula)
    while _parenthesis_balance(formula) > 0:
        formula += ")"
    return formula


In [30]:
def _nltk_flatten_application(expression, application_type):
    arguments = []
    while isinstance(expression, application_type):
        arguments.append(expression.argument)
        expression = expression.function
    return expression, list(reversed(arguments))


def _folio_term_to_ast(term, application_type, variable_type, constant_type):
    if isinstance(term, variable_type):
        return "?" + str(term.variable.name)
    if isinstance(term, constant_type):
        return _causal_name(term.variable.name)
    if isinstance(term, application_type):
        function, arguments = _nltk_flatten_application(term, application_type)
        name = getattr(getattr(function, "variable", None), "name", None)
        if name is None:
            raise ValueError("Unsupported FOL function term")
        return (_causal_name(name),) + tuple(
            _folio_term_to_ast(x, application_type, variable_type, constant_type)
            for x in arguments
        )
    raise ValueError("Unsupported FOL term: " + type(term).__name__)


def folio_formula_to_ast(formula):
    from nltk.sem.logic import (
        Expression, AllExpression, ExistsExpression, AndExpression,
        OrExpression, ImpExpression, IffExpression, NegatedExpression,
        ApplicationExpression, IndividualVariableExpression, ConstantExpression,
        EqualityExpression
    )

    expression = Expression.fromstring(_normalize_folio_formula(formula))

    def convert(node):
        if isinstance(node, AllExpression):
            return ("all", "?" + str(node.variable.name), convert(node.term))
        if isinstance(node, ExistsExpression):
            return ("ex", "?" + str(node.variable.name), convert(node.term))
        if isinstance(node, NegatedExpression):
            return ("not", convert(node.term))
        if isinstance(node, AndExpression):
            return ("and", convert(node.first), convert(node.second))
        if isinstance(node, OrExpression):
            return ("or", convert(node.first), convert(node.second))
        if isinstance(node, ImpExpression):
            return ("imp", convert(node.first), convert(node.second))
        if isinstance(node, IffExpression):
            return ("iff", convert(node.first), convert(node.second))
        if isinstance(node, EqualityExpression):
            left = _folio_term_to_ast(
                node.first, ApplicationExpression,
                IndividualVariableExpression, ConstantExpression
            )
            right = _folio_term_to_ast(
                node.second, ApplicationExpression,
                IndividualVariableExpression, ConstantExpression
            )
            if left == right:
                return ("true",)
            return ("atom", "__eq__", (left, right))
        if isinstance(node, ApplicationExpression):
            function, arguments = _nltk_flatten_application(node, ApplicationExpression)
            name = getattr(getattr(function, "variable", None), "name", None)
            if name is None:
                raise ValueError("Unsupported FOL predicate")
            args = tuple(
                _folio_term_to_ast(x, ApplicationExpression, IndividualVariableExpression, ConstantExpression)
                for x in arguments
            )
            return ("atom", _causal_name(name), args)
        raise ValueError("Unsupported FOL expression: " + type(node).__name__)

    return convert(expression)


In [31]:
def split_fol_premises(premises_fol):
    if isinstance(premises_fol, (list, tuple)):
        output = []
        for item in premises_fol:
            output.extend(split_fol_premises(str(item)))
        return [item for item in output if item]

    text = str(premises_fol).strip()
    if not text:
        return []

    parts = []
    start = 0
    depth = 0
    i = 0
    while i < len(text):
        char = text[i]
        if char == "(":
            depth += 1
        elif char == ")":
            depth = max(0, depth - 1)
            j = i + 1
            while j < len(text) and text[j].isspace():
                j += 1
            if j < len(text):
                next_text = text[j:]
                quantified_start = re.match(
                    r"(?:all|exists)\s+[A-Za-z][A-Za-z0-9_]*\s*\.",
                    next_text,
                    flags=re.I
                ) is not None
                next_char = text[j]
                next_is_formula = next_char in "∀∃¬(" or quantified_start
                if not next_is_formula and next_char.isalpha():
                    next_is_formula = re.match(
                        r"[A-Za-z_][A-Za-z0-9_]*\s*\(",
                        next_text
                    ) is not None
                if next_is_formula:
                    part = text[start:i + 1].strip()
                    if part:
                        parts.append(part)
                    start = j
                    depth = 0
                    i = j
                    continue
        i += 1

    last = text[start:].strip()
    if last:
        parts.append(last)
    return parts


def folio_resolution_reason(premises_fol, conclusion_fol):
    premise_lines = split_fol_premises(premises_fol)
    if not premise_lines or not str(conclusion_fol).strip():
        return "Unknown", {"reason": "missing_formal_annotation"}, []

    state = FolState()
    kb = []
    try:
        for line in premise_lines:
            kb.extend(to_cnf(folio_formula_to_ast(line), state.counters))
        conclusion_ast = folio_formula_to_ast(conclusion_fol)
        negated_goal = to_cnf(("not", conclusion_ast), state.counters)
        positive_goal = to_cnf(conclusion_ast, state.counters)
    except Exception as error:
        return "Unknown", {"reason": "formal_parse_or_cnf_error", "error": str(error)}, []

    kb_for_goal = connected_clauses(kb, negated_goal)
    prover_entail = ResolutionProver()
    entails = prover_entail.prove(kb_for_goal, negated_goal)
    entail_trace = prover_entail.proof_lines() if entails is True else []

    kb_for_refutation = connected_clauses(kb, positive_goal)
    prover_refute = ResolutionProver()
    refutes = prover_refute.prove(kb_for_refutation, positive_goal)

    details = {
        "premise_count": len(premise_lines),
        "entailed": entails,
        "negation_entailed": refutes,
        "entail_clauses_generated": prover_entail.generated,
        "refutation_clauses_generated": prover_refute.generated,
        "entail_search_truncated": prover_entail.truncated,
        "refutation_search_truncated": prover_refute.truncated,
    }

    if entails is None or refutes is None:
        return "Unknown", details, entail_trace
    if entails and refutes:
        return "Inconsistent", details, entail_trace
    if entails:
        return "True", details, entail_trace
    if refutes:
        return "False", details, []
    return "Uncertain", details, []


In [32]:
def parse_counterbench_info(given_info):
    text = str(given_info).strip()
    probabilities = {
        _causal_name(variable): float(probability)
        for variable, probability in re.findall(
            r"([A-Za-z][A-Za-z0-9_-]*)\s*~\s*Bern\((0(?:\.\d*)?|1(?:\.0*)?|\.\d+)\)",
            text,
            flags=re.I
        )
    }

    observations = []
    observation_matches = list(re.finditer(r"\bWe observed\s+([^.;]+)", text, flags=re.I))
    for match in observation_matches:
        for item in re.split(r"\s+and\s+", match.group(1).strip(), flags=re.I):
            item = item.strip().strip(",")
            parsed = re.fullmatch(
                r"(not\s+)?([A-Za-z][A-Za-z0-9_-]*)(?:\s+(not\s+occurs?|occurs?))?",
                item,
                flags=re.I
            )
            if parsed:
                leading_not, variable, suffix = parsed.groups()
                value = not bool(leading_not) and not bool(suffix and suffix.lower().startswith("not"))
                observations.append((_causal_name(variable), value))

    causal_text = re.sub(r"\bWe observed\s+[^.;]+", " ", text, flags=re.I)
    causal_text = re.sub(
        r"[A-Za-z][A-Za-z0-9_-]*\s*~\s*Bern\([^)]*\)\.?", " ", causal_text, flags=re.I
    )
    causal_text = re.sub(r"^\s*We know that\s+", " ", causal_text, flags=re.I)

    atom_pattern = r"(?:not\s+)?[A-Za-z][A-Za-z0-9_-]*"
    clause_pattern = re.compile(
        rf"(?P<lhs>{atom_pattern}(?:\s+(?:and|or)\s+{atom_pattern})*(?:\s+together)?)\s+"
        rf"causes?\s+(?P<negative>not\s+)?(?P<target>[A-Za-z][A-Za-z0-9_-]*)",
        flags=re.I
    )

    equations = defaultdict(lambda: {"positive": [], "negative": []})
    variables = set(probabilities)
    matches = list(clause_pattern.finditer(causal_text))
    for index, match in enumerate(matches):
        lhs = re.sub(r"\s+together$", "", match.group("lhs").strip(), flags=re.I)
        targets = [(match.group("target"), bool(match.group("negative")))]
        segment_end = matches[index + 1].start() if index + 1 < len(matches) else len(causal_text)
        remainder = causal_text[match.end():segment_end]
        for extra in re.finditer(
            r"(?:(not)\s+)?([A-Za-z][A-Za-z0-9_-]*)\b",
            remainder,
            flags=re.I
        ):
            name = extra.group(2)
            if name.lower() not in {"and", "or", "not", "together"}:
                targets.append((name, bool(extra.group(1))))

        if re.search(r"\s+or\s+", lhs, flags=re.I):
            mechanisms = [[part.strip()] for part in re.split(r"\s+or\s+", lhs, flags=re.I)]
        else:
            mechanisms = [[part.strip() for part in re.split(r"\s+and\s+", lhs, flags=re.I)]]

        signed_mechanisms = []
        for mechanism in mechanisms:
            signed_parents = []
            for item in mechanism:
                parent_match = re.fullmatch(r"(not\s+)?([A-Za-z][A-Za-z0-9_-]*)", item, flags=re.I)
                if not parent_match:
                    raise ValueError("Unsupported causal parent: " + item)
                is_positive = not bool(parent_match.group(1))
                parent = _causal_name(parent_match.group(2))
                signed_parents.append((parent, is_positive))
                variables.add(parent)
            signed_mechanisms.append(tuple(signed_parents))

        for target_name, negative_target in targets:
            target = _causal_name(target_name)
            sign = "negative" if negative_target else "positive"
            equations[target][sign].extend(signed_mechanisms)
            variables.add(target)

    variables.update(variable for variable, _ in observations)
    if not equations:
        raise ValueError("No supported causal rules found in given_info")

    return dict(equations), probabilities, observations, variables


In [33]:
def parse_counterbench_query(question, variables):
    question = str(question).strip()
    match = re.search(r"\bWould\s+([A-Za-z][A-Za-z0-9_-]*)\s+occur\b", question, flags=re.I)
    if not match:
        raise ValueError("Could not identify the CounterBench query target")
    target = _causal_name(match.group(1))

    lower = question.lower()
    if "assume" in lower:
        start = lower.find("assume") + len("assume")
        segment = question[start:]
        end = re.search(r"\bWould\b", segment, flags=re.I)
        if end:
            segment = segment[:end.start()]
    elif re.search(r"\bif\b", lower):
        start = re.search(r"\bif\b", lower).end()
        segment = question[start:]
        end = re.search(r"\binstead of\b|\?", segment, flags=re.I)
        if end:
            segment = segment[:end.start()]
    else:
        raise ValueError("No intervention clause found")

    interventions = {}
    for variable in variables:
        if re.search(r"\bnot\s+" + re.escape(variable) + r"\b", segment, flags=re.I):
            interventions[variable] = False
        elif re.search(r"\b" + re.escape(variable) + r"\b", segment, flags=re.I):
            interventions[variable] = True

    if not interventions:
        raise ValueError("No recognized interventions found")
    return target, interventions


In [34]:
def counterbench_topological_order(equations):
    targets = set(equations)
    indegree = {target: 0 for target in targets}
    children = defaultdict(set)
    for target, equation in equations.items():
        for mechanism in equation["positive"] + equation["negative"]:
            for parent, _ in mechanism:
                if parent in targets and target not in children[parent]:
                    children[parent].add(target)
                    indegree[target] += 1

    queue = deque(sorted(node for node, degree in indegree.items() if degree == 0))
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for child in sorted(children[node]):
            indegree[child] -= 1
            if indegree[child] == 0:
                queue.append(child)
    if len(order) != len(targets):
        raise ValueError("Cyclic causal graph is outside the supported SCM fragment")
    return order



def _counterbench_node_value(node, equation, state, noise_values):
    def active(mechanism):
        return all(state.get(parent, False) == expected for parent, expected in mechanism)

    positive = equation["positive"]
    negative = equation["negative"]
    positive_active = any(active(mechanism) for mechanism in positive)
    negative_active = any(active(mechanism) for mechanism in negative)

    if node in noise_values:
        baseline = bool(noise_values[node])
    elif not positive and negative:
        baseline = True
    else:
        baseline = False

    value = baseline or positive_active
    if negative:
        value = value and not negative_active
    return bool(value)


In [35]:
def simulate_counterbench(equations, variables, noise_values, interventions, return_all=False, max_cycle_variables=18):
    targets = set(equations)
    roots = set(variables) - targets
    base_state = {variable: bool(noise_values.get(variable, True)) for variable in roots}
    base_state.update({variable: bool(value) for variable, value in interventions.items()})

    try:
        order = counterbench_topological_order(equations)
    except ValueError:
        order = None

    if order is not None:
        state = dict(base_state)
        for node in order:
            if node in interventions:
                continue
            state[node] = _counterbench_node_value(node, equations[node], state, noise_values)
        return [state] if return_all else state

    endogenous = sorted(targets - set(interventions))
    if len(endogenous) > max_cycle_variables:
        raise ValueError("Cyclic equilibrium search budget exceeded")

    equilibria = []
    for values in itertools.product((False, True), repeat=len(endogenous)):
        candidate = dict(base_state)
        candidate.update(zip(endogenous, values))
        valid = True
        for node in endogenous:
            expected = _counterbench_node_value(node, equations[node], candidate, noise_values)
            if candidate[node] != expected:
                valid = False
                break
        if valid:
            equilibria.append(candidate)

    if not equilibria:
        raise ValueError("Cyclic SCM has no fixed-point equilibrium")
    if return_all:
        return equilibria
    if len(equilibria) != 1:
        raise ValueError("Cyclic SCM has multiple fixed-point equilibria")
    return equilibria[0]


In [36]:
def counterbench_reason(record, max_noise_variables=18):
    equations, probabilities, observations, variables = parse_counterbench_info(record["given_info"])
    target, interventions = parse_counterbench_query(record["question"], variables)
    if target not in variables:
        raise ValueError("Query target is absent from the causal model")
    if any(variable not in variables for variable in interventions):
        raise ValueError("Intervention variable is absent from the causal model")

    roots = set(variables) - set(equations)
    probability_variables = set(probabilities)

    def evaluate_assignments(noise_variables, root_prior):
        if len(noise_variables) > max_noise_variables:
            raise ValueError("Exact abduction budget exceeded")

        compatible_worlds = 0
        total_weight = 0.0
        positive_weight = 0.0
        outcomes = set()

        for bits in itertools.product((False, True), repeat=len(noise_variables)):
            noise_values = {}
            weight = 1.0
            for variable, value in zip(noise_variables, bits):
                if variable in probabilities:
                    probability = probabilities[variable]
                else:
                    probability = root_prior
                noise_values[variable] = value
                weight *= probability if value else (1.0 - probability)
            if weight <= 0:
                continue

            try:
                factual_states = simulate_counterbench(
                    equations, variables, noise_values, {}, return_all=True
                )
            except ValueError:
                continue

            compatible_factual = [
                state for state in factual_states
                if all(state.get(variable, False) == value for variable, value in observations)
            ]
            if not compatible_factual:
                continue

            try:
                counterfactual_states = simulate_counterbench(
                    equations, variables, noise_values, interventions, return_all=True
                )
            except ValueError:
                continue

            counterfactual_outcomes = {
                bool(state.get(target, False)) for state in counterfactual_states
            }
            if len(counterfactual_outcomes) > 1:
                raise ValueError("Counterfactual target is not unique across cyclic equilibria")

            outcome = next(iter(counterfactual_outcomes))
            compatible_worlds += 1
            total_weight += weight
            if outcome:
                positive_weight += weight
            outcomes.add(outcome)

        return compatible_worlds, total_weight, positive_weight, outcomes

    primary_variables = sorted(probability_variables)
    result = evaluate_assignments(primary_variables, 1.0)
    abduction_mode = "specified_exogenous_assignments"

    if result[1] == 0.0:
        expanded_variables = sorted(probability_variables | roots)
        result = evaluate_assignments(expanded_variables, 0.5)
        abduction_mode = "expanded_root_abduction"

    compatible_worlds, total_weight, positive_weight, outcomes = result
    if total_weight <= 0:
        raise ValueError("No exogenous assignment is compatible with the observations")

    probability_yes = positive_weight / total_weight
    prediction = probability_yes >= 0.5
    return prediction, {
        "question_type": str(record.get("type", "unknown")).lower(),
        "compatible_worlds": compatible_worlds,
        "posterior_probability_yes": probability_yes,
        "possible_counterfactual_outcomes": sorted(outcomes),
        "interventions": interventions,
        "observations": observations,
        "noise_variables": len(primary_variables),
        "root_variables": sorted(roots),
        "abduction_mode": abduction_mode,
        "rule_count": sum(len(v["positive"]) + len(v["negative"]) for v in equations.values()),
    }



## 8. The reasoning agent (router)

The agent exposes one prediction interface and selects among three classical reasoning modules: Horn forward/backward chaining for RuleTaker-style inputs, first-order resolution for FOLIO-style inputs, and structural causal model reasoning for CounterBench-style inputs. Routing uses the fields and content supplied with each example; the caller does not need to provide a dataset name. The public evaluation removes source labels and gold answers before prediction. Unsupported or unresolved cases remain unresolved rather than being silently converted to False.

In [37]:
class ReasoningAgent:
    def __init__(self, engine="forward", horn_threshold=0.98):
        self.engine = engine
        self.horn_threshold = horn_threshold
        self.route_counts = Counter()

    def looks_folio(self, sentences):
        if not sentences:
            return False
        tokens = [set(normalize_sentence(s).replace(",", " ").split()) for s in sentences]
        total = sum(len(x) for x in tokens)
        return any(words & FOLIO_MARKERS for words in tokens) or total / max(1, len(sentences)) > 14

    def split_input(self, context, question):
        context = "" if context is None else context
        question = "" if question is None else question
        if isinstance(context, (list, tuple)):
            context = "\n".join(str(x) for x in context)
        if not str(context).strip():
            parts = split_sentences(question)
            if len(parts) > 1:
                return " ".join(parts[:-1]), parts[-1]
        return str(context), str(question)

    def solve_causal(self, context, question):
        text = str(context) + " " + str(question)
        if not any(marker in text.lower() for marker in CAUSAL_MARKERS):
            return None
        outcome = causal_answer(text, question)
        if outcome is None:
            return None
        label, route = outcome
        trace = ["causal path: " + " -> ".join(route)] if route else ["no directed path between the variables"]
        return {"label": label, "route": "causal-bfs-direct-effect", "trace": trace}

    def solve_horn(self, context, question, want_trace=False):
        sentences = split_sentences(context)
        if self.looks_folio(sentences):
            return None
        kb = HornKnowledgeBase(context, negation_mode="naf")
        if not (kb.facts or kb.rules) or kb.parse_quality() < self.horn_threshold:
            return None
        alternatives = parse_horn_query(question)
        if alternatives is None:
            return None
        label, trace, stats = kb.answer(alternatives, self.engine, want_trace)
        stats.update({"parsed_sentences": kb.parse_total, "parse_failures": len(kb.parse_failures), "parse_quality": kb.parse_quality()})
        return {"label": label, "route": "horn-" + self.engine + "-naf", "trace": trace, "stats": stats, "status": "ok"}

    def solve_fol(self, context, question, want_trace=False):
        state = FolState()
        kb_clauses = []
        skipped = 0
        for sentence in split_sentences(context):
            formula = parse_fol_sentence(sentence, state)
            if formula is None:
                skipped += 1
                continue
            try:
                kb_clauses.extend(to_cnf(formula, state.counters))
            except OverflowError:
                skipped += 1

        goal_formula = parse_fol_sentence(question, state)
        if goal_formula is None:
            return {"label": None, "route": "fol-unparsed-goal", "trace": [], "status": "unparsed", "stats": {"skipped": skipped}}

        try:
            negated_goal = to_cnf(("not", goal_formula), state.counters)
            positive_goal = to_cnf(goal_formula, state.counters)
        except OverflowError:
            return {"label": None, "route": "fol-cnf-budget", "trace": [], "status": "budget_exceeded", "stats": {"skipped": skipped}}

        prover_entail = ResolutionProver()
        entails = prover_entail.prove(connected_clauses(kb_clauses, negated_goal), negated_goal)
        trace = prover_entail.proof_lines() if entails is True and want_trace else []

        prover_refute = ResolutionProver()
        refutes = prover_refute.prove(connected_clauses(kb_clauses, positive_goal), positive_goal)

        if entails is True and refutes is True:
            return {"label": None, "route": "fol-resolution", "trace": trace, "status": "inconsistent", "logical_status": "Inconsistent"}
        if entails is True:
            status, label = "True", True
        elif refutes is True:
            status, label = "False", False
        elif entails is None or refutes is None or skipped:
            status, label = "Unknown", None
        else:
            status, label = "Uncertain", False

        return {
            "label": label,
            "route": "fol-resolution",
            "trace": trace,
            "status": "ok" if label is not None else "unknown",
            "logical_status": status,
            "stats": {
                "skipped": skipped,
                "entail_generated": prover_entail.generated,
                "refute_generated": prover_refute.generated,
                "entail_truncated": prover_entail.truncated,
                "refute_truncated": prover_refute.truncated,
            },
        }

    def solve_folio_formal(self, example, want_trace=False):
        status, details, trace = folio_resolution_reason(
            example.get("premises-FOL"),
            example.get("conclusion-FOL")
        )
        if status == "True":
            label, result_status = True, "ok"
        elif status in ("False", "Uncertain"):
            label, result_status = False, "ok"
        elif status == "Inconsistent":
            label, result_status = None, "inconsistent"
        else:
            label, result_status = None, "unknown"
        return {
            "label": label,
            "route": "folio-formal-resolution",
            "trace": trace if want_trace else [],
            "status": result_status,
            "logical_status": status,
            "stats": details,
        }

    def predict_detailed(self, context, question, want_trace=False):
        context, question = self.split_input(context, question)
        result = self.solve_causal(context, question)
        if result is None:
            try:
                result = self.solve_horn(context, question, want_trace)
            except (RecursionError, SearchBudgetExceeded):
                result = None
        if result is None:
            try:
                result = self.solve_fol(context, question, want_trace)
            except RecursionError:
                result = {"label": None, "route": "fol-recursion-limit", "trace": [], "status": "budget_exceeded"}
        if result.get("label") is None and "status" not in result:
            result["status"] = "unknown"
        self.route_counts[result["route"]] += 1
        return result

    def predict_example(self, example, dataset_name=None, want_trace=False):
        dataset = (dataset_name or example.get("dataset") or example.get("source") or "auto").lower()
        if dataset == "auto":
            if example.get("given_info") is not None:
                dataset = "counterbench"
            elif example.get("premises-FOL") is not None or example.get("conclusion-FOL") is not None:
                dataset = "folio"
            elif "premises" in example and "conclusion" in example and "context" not in example:
                dataset = "folio"
            else:
                context_text = str(example.get("context", ""))
                question_text = str(example.get("question", ""))
                causal_question = (
                    re.search(r"\bcauses?\b", context_text, flags=re.I)
                    and re.search(r"\bwould\b", question_text, flags=re.I)
                    and re.search(r"\b(if|assume)\b", question_text, flags=re.I)
                )
                formal_syntax = bool(re.search(r"[∀∃⊕→↔¬∧∨]", context_text + " " + question_text))
                if causal_question:
                    dataset = "counterbench"
                elif formal_syntax:
                    dataset = "folio"
                else:
                    dataset = "auto"
        if "counterbench" in dataset or "backdoor" in dataset or "alpha_v1" in dataset:
            prediction, details = counterbench_reason(example)
            self.route_counts["counterbench-scm"] += 1
            return {"prediction": bool(prediction), "status": "ok", "route": "counterbench-scm", "details": details}
        if "folio" in dataset:
            if example.get("premises-FOL") and example.get("conclusion-FOL"):
                result = self.solve_folio_formal(example, want_trace)
                self.route_counts[result.get("route", "folio-formal-resolution")] += 1
            else:
                result = self.solve_fol(
                    example.get("context", example.get("premises", "")),
                    example.get("question", example.get("conclusion", "")),
                    want_trace
                )
            return {
                "prediction": result.get("label"),
                "status": result.get("status", "unknown"),
                "logical_status": result.get("logical_status"),
                "route": result.get("route"),
                "details": result.get("stats", {}),
                "trace": result.get("trace", []),
            }
        result = self.predict_detailed(example.get("context", ""), example.get("question", ""), want_trace)
        return {
            "prediction": result.get("label"),
            "status": result.get("status", "unknown"),
            "route": result.get("route"),
            "logical_status": result.get("logical_status"),
            "details": result.get("stats", {}),
            "trace": result.get("trace", []),
        }

    def predict(self, context, question):
        result = self.predict_detailed(context, question)
        return result.get("label")


## 9. Data loading, evaluation and submission writer

Dataset repository names and column names differ between Hugging Face mirrors. Edit `DATASET_SOURCES` or point `LOCAL_FILES` at downloaded CSV / JSON / JSONL / Parquet files if automatic loading fails. Set `TEST_FILE` to the test file when it is available.

In [38]:
DATASET_SOURCES = {
    "ruletaker": "tasksource/ruletaker",
    "folio": "tasksource/folio",
}
COUNTERBENCH_FILES = ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]
LOCAL_FILES = {}
TEST_FILE = None
TEST_DATASET = "auto"
OUTPUT_DIR = "submission"
MAX_EXAMPLES_PER_DATASET = None

CONTEXT_KEYS = ["context", "premises", "premise", "theory", "story", "knowledge_base", "kb", "passage", "text", "facts", "given_info"]
QUERY_KEYS = ["question", "statement", "conclusion", "hypothesis", "query", "claim", "sentence"]
LABEL_KEYS = ["label", "answer", "flag", "gold", "target", "output"]
ID_KEYS = ["id", "example_id", "idx", "index", "uid", "story_id", "question_id"]
TRUE_VALUES = {"true", "entailment", "entailed", "yes", "1", "t", "correct"}


def normalize_label(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    return str(value).strip().lower() in TRUE_VALUES


def first_key(row, keys):
    lowered = {str(key).lower(): key for key in row}
    for key in keys:
        if key.lower() in lowered:
            return lowered[key.lower()]
    return None


In [39]:
def row_to_example(row, position, source):
    source_name = str(source).lower()
    ck = first_key(row, CONTEXT_KEYS)
    qk = first_key(row, QUERY_KEYS)
    lk = first_key(row, LABEL_KEYS)
    ik = first_key(row, ID_KEYS)

    if source_name == "auto":
        dataset_key = first_key(row, ["dataset", "dataset_name", "source", "task"])
        if dataset_key is not None and str(row[dataset_key]).strip():
            source_name = str(row[dataset_key]).strip().lower()
        elif first_key(row, ["given_info"]) is not None:
            source_name = "counterbench"
        elif first_key(row, ["premises-FOL", "conclusion-FOL"]) is not None:
            source_name = "folio"
        elif (
            first_key(row, ["premises"]) is not None
            and first_key(row, ["conclusion"]) is not None
            and first_key(row, ["context"]) is None
        ):
            source_name = "folio"
        else:
            context_text = str(row[ck]) if ck else ""
            question_text = str(row[qk]) if qk else ""
            combined_text = context_text + " " + question_text
            causal_question = (
                re.search(r"\bcauses?\b", context_text, flags=re.I)
                and re.search(r"\bwould\b", question_text, flags=re.I)
                and re.search(r"\b(if|assume)\b", question_text, flags=re.I)
            )
            formal_syntax = bool(re.search(r"[∀∃⊕→↔¬∧∨]", combined_text))
            folio_language = bool(re.search(
                r"\b(either|neither|nor|unless|only if|iff|both|most|least|"
                r"at least|exactly one|there exists|there is|there are)\b",
                combined_text,
                flags=re.I
            ))
            if causal_question:
                source_name = "counterbench"
            elif formal_syntax:
                source_name = "folio"
            elif folio_language:
                source_name = "folio"
            else:
                source_name = "auto"

    context = row[ck] if ck else ""
    if isinstance(context, (list, tuple)):
        context = "\n".join(str(item) for item in context)
    question = row[qk] if qk else ""
    if ck == qk:
        context = ""

    raw_label = row[lk] if lk else None
    identifier = row[ik] if ik and row[ik] is not None else position
    example = {
        "id": identifier,
        "context": context,
        "question": question,
        "label": normalize_label(raw_label),
        "label_raw": raw_label,
        "dataset": source_name,
        "source": source_name,
        "raw": dict(row),
    }
    for key in ("given_info", "premises", "conclusion", "premises-FOL", "conclusion-FOL", "type", "meta"):
        original_key = first_key(row, [key])
        if original_key is not None:
            example[key] = row[original_key]
    if source_name == "counterbench":
        example["given_info"] = example.get("given_info", context)
        example["context"] = example["given_info"]
        example["question"] = row[qk] if qk else question
    if source_name == "folio":
        example["context"] = example.get("premises", context)
        example["question"] = example.get("conclusion", question)
    return example


def _load_hf_split(repo_id, preferred_splits, source, limit=None):
    from datasets import load_dataset
    dataset = load_dataset(repo_id)
    split_name = next((name for name in preferred_splits if name in dataset), None)
    if split_name is None:
        raise ValueError("No public development split found for " + repo_id)
    split = dataset[split_name]
    total = len(split) if limit is None else min(len(split), int(limit))
    examples = [row_to_example(split[index], index, source) for index in range(total)]
    print("loaded %s from %s[%s]: %d examples" % (source, repo_id, split_name, len(examples)))
    return examples


def load_counterbench(limit=None):
    from huggingface_hub import hf_hub_download
    examples = []
    for filename in COUNTERBENCH_FILES:
        path = hf_hub_download(
            repo_id="CounterBench/CounterBench",
            filename=filename,
            repo_type="dataset"
        )
        with open(path, encoding="utf-8") as handle:
            records = json.load(handle)
        if isinstance(records, dict):
            records = records.get("data", records.get("examples", []))
        for index, record in enumerate(records):
            example = row_to_example(record, "%s:%s" % (filename, index), "counterbench")
            example["source_file"] = filename
            examples.append(example)
            if limit is not None and len(examples) >= int(limit):
                return examples
        print("loaded counterbench file %s: %d examples" % (filename, len(records)))
    return examples


def load_public_datasets(limit=MAX_EXAMPLES_PER_DATASET):
    public = {}
    if "ruletaker" in LOCAL_FILES and os.path.exists(LOCAL_FILES["ruletaker"]):
        public["ruletaker"] = load_local(LOCAL_FILES["ruletaker"], "ruletaker", limit)
    else:
        public["ruletaker"] = _load_hf_split(DATASET_SOURCES["ruletaker"], ("dev", "validation"), "ruletaker", limit)

    if "folio" in LOCAL_FILES and os.path.exists(LOCAL_FILES["folio"]):
        public["folio"] = load_local(LOCAL_FILES["folio"], "folio", limit)
    else:
        public["folio"] = _load_hf_split(DATASET_SOURCES["folio"], ("validation", "dev"), "folio", limit)

    if "counterbench" in LOCAL_FILES and os.path.exists(LOCAL_FILES["counterbench"]):
        public["counterbench"] = load_local(LOCAL_FILES["counterbench"], "counterbench", limit)
    else:
        public["counterbench"] = load_counterbench(limit)
    return {name: examples for name, examples in public.items() if examples}


In [40]:
def load_local(path, name="auto", limit=None):
    lower = str(path).lower()
    if lower.endswith(".csv"):
        with open(path, newline="", encoding="utf-8") as handle:
            records = list(csv.DictReader(handle))
    elif lower.endswith(".jsonl"):
        with open(path, encoding="utf-8") as handle:
            records = [json.loads(line) for line in handle if line.strip()]
    elif lower.endswith(".json"):
        with open(path, encoding="utf-8") as handle:
            data = json.load(handle)
        records = data if isinstance(data, list) else data.get("data", data.get("examples", []))
    elif lower.endswith(".parquet"):
        if pd is None:
            raise ImportError("pandas is required for parquet files")
        records = pd.read_parquet(path).to_dict("records")
    else:
        raise ValueError("Supported private test formats: CSV, JSON, JSONL, Parquet")
    if limit is not None:
        records = records[:int(limit)]
    return [row_to_example(record, index, name) for index, record in enumerate(records)]


## 10. Public evaluation without dataset names\n\nThe evaluator still knows which public split a row came from so that accuracy can be calculated. The prediction call receives a fresh example with dataset/source metadata and gold labels removed. The agent has to route from the remaining input structure and content.

In [41]:
def make_blind_example(example, position=0):
    raw = example.get("raw")
    if isinstance(raw, dict):
        row = dict(raw)
    else:
        ignored = {"dataset", "dataset_name", "source", "task", "source_file", "label", "label_raw", "raw", "prediction"}
        row = {key: value for key, value in example.items() if str(key).lower() not in ignored}

    hidden_fields = {"dataset", "dataset_name", "source", "task", "source_file"}
    target_fields = {key.lower() for key in LABEL_KEYS}
    for key in list(row):
        lowered = str(key).lower()
        if lowered in hidden_fields or lowered in target_fields:
            row.pop(key, None)

    return row_to_example(row, position, "auto")


def route_family(route):
    route = str(route or "").lower()
    if "counterbench" in route or "causal" in route:
        return "counterbench"
    if route.startswith("folio") or route.startswith("fol-") or "resolution" in route:
        return "folio"
    if "horn" in route:
        return "ruletaker"
    return "unknown"


In [42]:
def evaluate_dataset(agent, examples, dataset_name, show_errors=5):
    rows = []
    started = time.perf_counter()
    for index, example in enumerate(examples):
        query_start = time.perf_counter()
        blind_example = make_blind_example(example, index)
        try:
            output = agent.predict_example(blind_example)
            prediction = output.get("prediction")
            status = output.get("status", "unknown")
            logical_status = output.get("logical_status")
            error = None
        except Exception as exception:
            prediction, status, logical_status = None, "error", None
            output, error = {}, str(exception)
        elapsed = time.perf_counter() - query_start

        actual_raw = example.get("label_raw")
        actual_binary = example.get("label")
        selected_module = route_family(output.get("route"))
        row = {
            "id": example.get("id", index),
            "dataset": dataset_name,
            "prediction": prediction,
            "status": status,
            "logical_status": logical_status,
            "selected_module": selected_module,
            "route_matches_dataset": selected_module == dataset_name,
            "actual_binary": actual_binary,
            "actual_raw": actual_raw,
            "correct_binary": (prediction == actual_binary) if prediction is not None and actual_binary is not None else None,
            "reasoning_time": elapsed,
            "details": output.get("details", {}),
            "error": error,
        }
        if dataset_name == "folio":
            row["correct_3class"] = (logical_status == str(actual_raw).strip().capitalize()) if logical_status in ("True", "False", "Uncertain") else None
        rows.append(row)

    total_elapsed = time.perf_counter() - started
    results = pd.DataFrame(rows) if pd is not None else rows
    if pd is None:
        completed = [r for r in rows if r["prediction"] is not None]
        correct = [r for r in completed if r["correct_binary"]]
        print("%s: %d/%d completed, binary accuracy %.2f%%, %.3fs" % (dataset_name, len(completed), len(rows), 100 * len(correct) / max(1, len(completed)), total_elapsed))
        return results

    completed = results[results["prediction"].notna()]
    print("\n%s" % dataset_name.upper())
    print("Total:", len(results))
    print("Completed:", len(completed))
    print("Coverage (%):", round(100 * len(completed) / max(1, len(results)), 2))
    print("Binary accuracy on completed (%):", round(100 * completed["correct_binary"].mean(), 2) if len(completed) else float("nan"))
    print("Coverage-adjusted correct rate (%):", round(100 * results["correct_binary"].fillna(False).mean(), 2))
    print("Automatic route match (%):", round(100 * results["route_matches_dataset"].mean(), 2))
    print("Total time (s):", round(total_elapsed, 3))
    print("Average time (ms/query):", round(1000 * total_elapsed / max(1, len(results)), 4))
    print("Statuses:", results["status"].value_counts().to_dict())

    if dataset_name == "folio":
        folio_done = results[results["correct_3class"].notna()]
        print("FOLIO 3-class accuracy (%):", round(100 * folio_done["correct_3class"].mean(), 2) if len(folio_done) else float("nan"))

    mistakes = results[results["correct_binary"] == False].head(show_errors)
    for _, row in mistakes.iterrows():
        print("miss:", row["id"], "prediction:", row["prediction"], "actual:", row["actual_raw"], "route:", row["selected_module"], "status:", row["status"])
    return results


In [43]:
def write_submission(agent, examples, path, default_dataset="auto"):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)

    started = time.perf_counter()
    rows = []
    unresolved = []
    for index, example in enumerate(examples):
        blind_example = make_blind_example(example, index)
        output = agent.predict_example(blind_example)
        prediction = output.get("prediction")
        identifier = example.get("id", index)
        if prediction is None:
            unresolved.append((identifier, output.get("status"), output.get("details", {})))
            continue
        rows.append((identifier, "True" if bool(prediction) else "False"))

    elapsed = time.perf_counter() - started
    if unresolved:
        preview = unresolved[:10]
        raise RuntimeError(
            "Submission not written: %d/%d examples are unresolved. "
            "Resolve these cases rather than assigning an unsupported False label. "
            "First unresolved examples: %s" % (len(unresolved), len(examples), preview)
        )

    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "label"])
        writer.writerows(rows)

    print("wrote %d predictions to %s in %.3fs; unresolved: 0" % (len(rows), path, elapsed))
    return rows, elapsed, 0


## 10. Self tests and trace printer

In [44]:
SELF_TESTS = [
    ("Bob is young. Bob is kind. If someone is young then they are smart. If someone is smart and kind then they are successful.", "Bob is successful.", True),
    ("Bob is young. Bob is kind. If someone is young then they are smart. If someone is smart and kind then they are successful.", "Bob is not successful.", False),
    ("The cat is big. The cat likes the dog. The dog is red. If something is big and not red then it is nice. Big, red things are rough.", "The dog is rough.", False),
    ("The cat is big. The cat likes the dog. If something is big and not red then it is nice.", "The cat is nice.", True),
    ("The bald eagle chases the cat. If something chases the cat then it is green. All green things are cold.", "The bald eagle is cold.", True),
    ("Anne is big. Anne is not red. If something is not red then it is blue.", "Anne is blue.", True),
    ("Anne is big. Anne is not red. If something is big and not red then it is blue.", "Anne is blue.", True),
    ("Anne is big. If something is big and not red then it is blue.", "Anne is not blue.", False),
    ("All students are happy. Tom is a student. Mary is a student.", "Mary is happy.", True),
    ("All people who study hard are successful. Mia studies hard.", "Mia is successful.", True),
    ("If a person is rich, then they are happy. Ann is rich.", "Ann is happy.", True),
    ("Some cats are black. All black things are dark.", "Some cats are dark.", True),
    ("Tom is either happy or sad. Tom is not happy.", "Tom is sad.", True),
    ("No animals are plants. Rex is an animal.", "Rex is not a plant.", True),
    ("All birds can fly. Tweety is a bird.", "Tweety can fly.", True),
    ("Tom is happy.", "Tom is sad.", False),
    ("Rain has a direct effect on wet ground. Wet ground has a direct effect on slipping. Sun has a direct effect on tan.", "If we intervene on rain, would slipping change?", True),
    ("Dave is not cold. Dave is kind. Dave is round. If something is cold then it is not smart. If something is not furry then it is smart. If something is green then it is not furry.", "Dave is smart.", True),
    ("Anne is nice. Bob is green. Bob is red. Bob is rough. Charlie is kind. Harry is red. Harry is rough. If something is rough and not nice then it is red. Red, kind things are big. Big things are green. All big, kind things are rough. All rough, nice things are big. All nice, big things are green. All nice things are kind. If something is big and green then it is smart. If something is kind then it is rough.", "Charlie is big.", True),
    ("The bear sees the cat. The cat is not big. The cat likes the bear. The cat sees the squirrel. The cow is green. The cow does not like the squirrel. The squirrel likes the bear. The squirrel likes the cow. The squirrel sees the bear. The squirrel sees the cat. If someone visits the squirrel and the squirrel likes the cow then they visit the bear. If someone visits the cow and they do not like the bear then they are blue. All green people are blue. If someone visits the squirrel and they visit the bear then they are not nice. If someone likes the cow and they do not like the cat then they are green. If someone likes the cat then they are rough. If someone visits the bear and they see the cat then the cat sees the bear. If someone sees the squirrel and the squirrel is blue then they like the cow.", "The cat is green.", True),
    ("Charlie is red.", "Charlie is not quiet.", True),
    ("Charlie is quiet.", "Charlie is not quiet.", False),
]


In [45]:
def run_self_tests(engine="forward"):
    agent = ReasoningAgent(engine=engine)
    passed = 0
    for context, question, expected in SELF_TESTS:
        result = agent.predict_detailed(context, question)
        ok = bool(result["label"]) == expected
        passed += ok
        print("%-4s %-22s %s" % ("ok" if ok else "FAIL", result["route"], question))
    print("%d/%d self tests passed (%s engine)" % (passed, len(SELF_TESTS), engine))
    return passed == len(SELF_TESTS)


def show_traces(agent):
    demos = [
        ("Bob is young. Bob is kind. If someone is young then they are smart. If someone is smart and kind then they are successful.", "Bob is successful."),
        ("The cat is big. The cat likes the dog. If something is big and not red then it is nice.", "The cat is nice."),
        ("All students are happy. Tom is a student.", "Tom is happy."),
        ("Everyone who likes music is a dancer. Bob likes music. Some cats are black.", "Bob is a dancer."),
    ]
    for context, question in demos:
        result = agent.predict_detailed(context, question, want_trace=True)
        print("QUERY :", question)
        print("ROUTE :", result["route"], "->", result["label"])
        for line in result["trace"]:
            print("   ", line)
        print()


## 11. Synthetic benchmark generator and engine comparison

In [46]:
SYN_NAMES = ["Anne", "Bob", "Charlie", "Dave", "Erin", "Fiona"]
SYN_ADJECTIVES = ["big", "red", "blue", "kind", "nice", "round", "cold", "rough", "smart", "young"]


def synthetic_problem(rng, owa=False):
    sentences = []
    for name in SYN_NAMES[:rng.randint(2, 5)]:
        for adj in rng.sample(SYN_ADJECTIVES, rng.randint(1, 3)):
            sentences.append("%s is %s%s." % (name, "not " if owa and rng.random() < 0.25 else "", adj))
    for _ in range(rng.randint(3, 8)):
        conditions = []
        for adj in rng.sample(SYN_ADJECTIVES, rng.randint(1, 2)):
            conditions.append(("not " if rng.random() < 0.2 else "") + adj)
        negate_head = "not " if owa and rng.random() < 0.2 else ""
        sentences.append("If someone is %s then they are %s%s." % (" and ".join(conditions), negate_head, rng.choice(SYN_ADJECTIVES)))
    for _ in range(rng.randint(0, 3)):
        sentences.append("%s things are %s." % (", ".join(rng.sample(SYN_ADJECTIVES, 2)).capitalize(), rng.choice(SYN_ADJECTIVES)))
    query = "%s is %s%s." % (rng.choice(SYN_NAMES), rng.choice(["", "not "]), rng.choice(SYN_ADJECTIVES))
    return " ".join(sentences), query


def engine_comparison(n=300, seed=RANDOM_SEED):
    rng = random.Random(seed)
    problems = [synthetic_problem(rng, owa=(i % 2 == 1)) for i in range(n)]
    agents = {"forward": ReasoningAgent(engine="forward"), "backward": ReasoningAgent(engine="backward")}
    answers = {}
    timings = {}
    for name, agent in agents.items():
        start = time.perf_counter()
        answers[name] = [agent.predict(c, q) for c, q in problems]
        timings[name] = time.perf_counter() - start
    comparable = [(a, b) for a, b in zip(answers["forward"], answers["backward"]) if a is not None and b is not None]
    agreement = sum(a == b for a, b in comparable)
    print("synthetic problems:", n)
    for name in agents:
        resolved = [value for value in answers[name] if value is not None]
        true_rate = sum(value is True for value in resolved) / max(1, len(resolved))
        print("  %-9s total %.3fs  (%.3f ms/query)  true-rate %.2f  resolved %d/%d" % (name, timings[name], 1000 * timings[name] / n, true_rate, len(resolved), n))
    print("  agreement between engines: %d/%d comparable (%d unresolved pairs)" % (agreement, len(comparable), n - len(comparable)))
    return timings, agreement


## 12. Run: self tests, engine comparison, evaluation on public data (accuracy and time)

In [47]:
agent = ReasoningAgent(engine="forward")

run_self_tests("forward")
run_self_tests("backward")
engine_comparison(300)


ok   horn-forward-naf       Bob is successful.
ok   horn-forward-naf       Bob is not successful.
ok   horn-forward-naf       The dog is rough.
ok   horn-forward-naf       The cat is nice.
ok   horn-forward-naf       The bald eagle is cold.
ok   horn-forward-naf       Anne is blue.
ok   horn-forward-naf       Anne is blue.
ok   horn-forward-naf       Anne is not blue.
ok   horn-forward-naf       Mary is happy.
ok   fol-resolution         Mia is successful.
ok   horn-forward-naf       Ann is happy.
ok   fol-resolution         Some cats are dark.
ok   fol-resolution         Tom is sad.
ok   fol-resolution         Rex is not a plant.
ok   fol-resolution         Tweety can fly.
ok   horn-forward-naf       Tom is sad.
ok   causal-bfs-direct-effect If we intervene on rain, would slipping change?
ok   horn-forward-naf       Dave is smart.
ok   horn-forward-naf       Charlie is big.
ok   horn-forward-naf       The cat is green.
ok   horn-forward-naf       Charlie is not quiet.
ok   horn-forwar

({'forward': 0.2149142859998392, 'backward': 0.1510789419990033}, 298)

In [48]:
counterbench_checks = [
    {
        "type": "basic",
        "given_info": "We know that X causes A, A causes Y.",
        "question": "Would Y occur if not X instead of X?",
        "answer": "no"
    },
    {
        "type": "conditional",
        "given_info": "We know that X and V1 together cause V2, V2 causes Y. We observed V1.",
        "question": "Would Y occur if not X instead of X?",
        "answer": "no"
    },
    {
        "type": "joint",
        "given_info": "We know that X causes V1, X and V1 together cause V2, V2 causes Y.",
        "question": "Would Y occur if not X and not V1?",
        "answer": "no"
    },
    {
        "type": "nested",
        "given_info": "We know that X causes V1, V1 causes V2, V2 causes Y.",
        "question": "Assume not X, and based on this assumption, further suppose not V2. Would Y occur?",
        "answer": "no"
    },
    {
        "type": "backdoor",
        "given_info": "We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune",
        "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
        "answer": "yes"
    },
    {
        "type": "backdoor",
        "given_info": "We know that Vork causes Ziklo, Ziklo causes not Blaf, Blaf causes Trune, and Trune and Vork together cause Lumbo. Blaf~Bern(0.6). We observed Trune not occurs",
        "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
        "answer": "yes"
    },
    {
        "type": "backdoor",
        "given_info": "We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune",
        "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
        "answer": "yes"
    },
    {
        "type": "backdoor",
        "given_info": "We know that Blaf causes Ziklo, Ziklo and Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune",
        "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
        "answer": "no"
    },
]

counterbench_test_rows = []
for example in counterbench_checks:
    predicted, details = counterbench_reason(example)
    expected = str(example["answer"]).lower() == "yes"
    counterbench_test_rows.append({
        "type": example["type"],
        "predicted": predicted,
        "expected": expected,
        "passed": predicted == expected,
        "probability_yes": details["posterior_probability_yes"],
    })

if pd is not None:
    display(pd.DataFrame(counterbench_test_rows))
else:
    print(counterbench_test_rows)
assert all(row["passed"] for row in counterbench_test_rows)


,type,predicted,expected,passed,probability_yes
0,basic,False,False,True,0.0
1,conditional,False,False,True,0.0
2,joint,False,False,True,0.0
3,nested,False,False,True,0.0
4,backdoor,True,True,True,1.0
5,backdoor,True,True,True,0.6
6,backdoor,True,True,True,1.0
7,backdoor,False,False,True,0.0


In [49]:
folio_formal_status, folio_formal_stats, _ = folio_resolution_reason(
    [
        "∀x (Human(x) → Mortal(x))",
        "Human(Socrates)"
    ],
    "Mortal(Socrates)"
)
assert folio_formal_status == "True"

joined_folio_status, _, _ = folio_resolution_reason(
    "∀x (Human(x) → Mortal(x)) Human(Socrates)",
    "Mortal(Socrates)"
)
assert joined_folio_status == "True"

folio_annotation_sample = [
    "∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))",
    "∀x (DrinkRegularly(x, coffee) ∨ (¬WantToBeAddictedTo(x, caffeine)))",
    "∀x (¬WantToBeAddictedTo(x, caffeine) → ¬AwareThatDrug(x, caffeine))",
    "¬(Student(rina) ⊕ ¬AwareThatDrug(rina, caffeine))",
    "¬(IsDependentOn(rina, caffeine) ⊕ Student(rina))",
]
folio_annotation_status, _, _ = folio_resolution_reason(
    " ".join(folio_annotation_sample),
    "¬WantToBeAddictedTo(rina, caffeine) ∨ (¬AwareThatDrug(rina, caffeine))"
)
assert folio_annotation_status == "True"

routing_tests = [
    ({"premises": "All humans are mortal. Socrates is human.", "conclusion": "Socrates is mortal.", "premises-FOL": "all x.(Human(x) -> Mortal(x)) Human(socrates)", "conclusion-FOL": "Mortal(socrates)", "label": "True"}, "folio"),
    ({"given_info": "We know that X causes Y.", "question": "Would Y occur if not X instead of X?", "answer": "no"}, "counterbench"),
    ({"context": "Bob is young.", "question": "Bob is young.", "label": "entailment"}, "ruletaker"),
    ({"context": "X causes A. A causes Y.", "question": "Would Y occur if not X instead of X?"}, "counterbench"),
]
for raw_row, expected_dataset in routing_tests:
    normalized = row_to_example(raw_row, 0, "auto")
    output = ReasoningAgent().predict_example(make_blind_example(normalized))
    assert route_family(output.get("route")) == expected_dataset
    assert output.get("prediction") is not None

print("CounterBench SCM regression tests: %d/%d passed" % (
    sum(row["passed"] for row in counterbench_test_rows),
    len(counterbench_test_rows)
))
print("FOLIO resolution regression test: passed")
print("Automatic dataset routing tests: %d/%d passed" % (len(routing_tests), len(routing_tests)))


CounterBench SCM regression tests: 8/8 passed
FOLIO resolution regression test: passed
Automatic dataset routing tests: 4/4 passed


In [50]:
negative_only_rule_check = (
    "The cat sees the bear. The mouse sees the cat. "
    "If someone does not see the bald eagle then the bald eagle is round."
)
negative_only_kb = HornKnowledgeBase(negative_only_rule_check, negation_mode="naf")
negative_only_answer, _, _ = negative_only_kb.answer(
    parse_horn_query("The bald eagle is round."), "forward", False
)
assert negative_only_answer is True

negative_query_missing = HornKnowledgeBase("Bob is kind.", negation_mode="naf")
missing_negative_answer, _, _ = negative_query_missing.answer(
    parse_horn_query("Bob is not smart."), "forward", False
)
assert missing_negative_answer is True

positive_query_blocks_negation = HornKnowledgeBase("Bob is smart.", negation_mode="naf")
blocked_negative_answer, _, _ = positive_query_blocks_negation.answer(
    parse_horn_query("Bob is not smart."), "forward", False
)
assert blocked_negative_answer is False

negative_query_explicit = HornKnowledgeBase("Bob is not smart.", negation_mode="naf")
explicit_negative_answer, _, _ = negative_query_explicit.answer(
    parse_horn_query("Bob is not smart."), "forward", False
)
assert explicit_negative_answer is True

folio_repair_samples = [
    "∀x (InThisClub(x) ∧ (((YoungChildren(x) ∨ Teenager(x)) ∧ ¬(YoungChildren(x) ∧ Teenager(x)))) ∧ WishToFurther(x, academicCareer)) → Studen(x) ∧ AttendSchool(x))",
    "∀x ((Game(x) ∧ ∃y (GreaterThan(y, oneMillion) ∧ CopiesSold(x, y))) → Top10(x)))",
]
for formula in folio_repair_samples:
    formula_ast = folio_formula_to_ast(formula)
    to_cnf(formula_ast, FolState().counters)

joined_quantified = split_fol_premises(
    "all x.(At(x, mixer) & (GrandSlam(x) & Champion(x) -> Professional(x) & TennisPlayer(x))) "
    "all x.(At(x, mixer) & (OscarNominated(x) & Actor(x) -> Celebrity(x))) "
    "At(djokovic, mixer)"
)
assert len(joined_quantified) == 3
for formula in joined_quantified:
    folio_formula_to_ast(formula)

multi_effect_info = (
    "We know that Vork causes Sline, Frim, and Qado, "
    "Frim or not Qado causes Jurf, and Jurf causes Lumbo."
)
multi_effect_equations, _, _, _ = parse_counterbench_info(multi_effect_info)
assert "sline" in multi_effect_equations and "frim" in multi_effect_equations and "qado" in multi_effect_equations

counterbench_root_abduction_check = {
    "type": "conditional",
    "given_info": "We know that Ziklo causes Blaf, Blaf and Trune together cause Vork, Vork causes Sline, and Sline causes Lumbo. We observed not Trune.",
    "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
    "answer": "no",
}
cb_root_prediction, cb_root_details = counterbench_reason(counterbench_root_abduction_check)
assert cb_root_details["root_variables"]
assert cb_root_details["compatible_worlds"] > 0

counterbench_cycle_check = {
    "type": "conditional",
    "given_info": "We know that Ziklo causes not Blaf, Ziklo causes Trune, Blaf and Trune and Vork together cause Sline, Trune or Sline cause not Vork, Vork causes Frim, and Frim causes Lumbo. We observed Blaf.",
    "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
    "answer": "yes",
}
cb_cycle_prediction, cb_cycle_details = counterbench_reason(counterbench_cycle_check)
assert cb_cycle_details["compatible_worlds"] > 0
assert cb_cycle_prediction is True

print("Additional regression checks: RuleTaker negative-only grounding, FOLIO parenthesis repair, CounterBench root abduction, cyclic SCM equilibrium")


Additional regression checks: RuleTaker negative-only grounding, FOLIO parenthesis repair, CounterBench root abduction, cyclic SCM equilibrium


## 12. Automatic routing checks\n\nThese checks remove dataset labels and exercise the same prediction entry point used by the public evaluation. The input field structure is retained because it is part of the example, not a dataset-name hint.

In [51]:
blind_routing_examples = [
    (
        "ruletaker",
        {"id": "rt-1", "context": "Bob is young. If someone is young then they are smart.", "question": "Bob is smart.", "label": "entailment"}
    ),
    (
        "folio",
        {"id": "fo-1", "premises": "All humans are mortal. Socrates is human.", "conclusion": "Socrates is mortal.", "premises-FOL": "all x.(Human(x) -> Mortal(x)) Human(socrates)", "conclusion-FOL": "Mortal(socrates)", "label": "True"}
    ),
    (
        "counterbench",
        {"id": "cb-1", "given_info": "We know that X causes A, and A causes Y.", "question": "Would Y occur if not X instead of X?", "answer": "no", "type": "basic"}
    ),
]

blind_routing_results = []
for expected_module, raw_example in blind_routing_examples:
    example = make_blind_example(raw_example, len(blind_routing_results))
    output = ReasoningAgent().predict_example(example)
    selected_module = route_family(output.get("route"))
    blind_routing_results.append({
        "expected_module": expected_module,
        "selected_module": selected_module,
        "prediction": output.get("prediction"),
        "status": output.get("status"),
        "passed": selected_module == expected_module and output.get("prediction") is not None,
    })

if pd is not None:
    display(pd.DataFrame(blind_routing_results))
else:
    print(blind_routing_results)

assert all(row["passed"] for row in blind_routing_results)
print("Automatic routing checks: %d/%d passed" % (
    sum(row["passed"] for row in blind_routing_results),
    len(blind_routing_results)
))


,expected_module,selected_module,prediction,status,passed
0,ruletaker,ruletaker,True,ok,True
1,folio,folio,True,ok,True
2,counterbench,counterbench,False,ok,True


Automatic routing checks: 3/3 passed


In [52]:
agent = ReasoningAgent(engine="forward")
public = load_public_datasets()
results_by_dataset = {}
for dataset_name, examples in public.items():
    results_by_dataset[dataset_name] = evaluate_dataset(agent, examples, dataset_name)

summary_rows = []
for dataset_name, frame in results_by_dataset.items():
    if pd is None:
        continue
    completed = frame[frame["prediction"].notna()]
    summary_rows.append({
        "dataset": dataset_name,
        "total": len(frame),
        "completed": len(completed),
        "coverage_pct": 100 * len(completed) / max(1, len(frame)),
        "binary_accuracy_pct": 100 * completed["correct_binary"].mean() if len(completed) else float("nan"),
        "coverage_adjusted_correct_pct": 100 * frame["correct_binary"].fillna(False).mean(),
        "auto_route_match_pct": 100 * frame["route_matches_dataset"].mean(),
        "total_inference_seconds": frame["reasoning_time"].sum(),
    })

if summary_rows:
    multidataset_summary = pd.DataFrame(summary_rows)
    display(multidataset_summary.round(3))

print("reasoning routes used:", dict(agent.route_counts))


/usr/lib/python3/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded ruletaker from tasksource/ruletaker[dev]: 75872 examples
loaded folio from tasksource/folio[validation]: 203 examples
loaded counterbench file data_balanced_alpha_V1.json: 1000 examples
loaded counterbench file data_balanced_backdoor_V2.json: 200 examples

RULETAKER
Total: 75872
Completed: 75872
Coverage (%): 100.0
Binary accuracy on completed (%): 100.0
Coverage-adjusted correct rate (%): 100.0
Automatic route match (%): 100.0
Total time (s): 46.293
Average time (ms/query): 0.6101
Statuses: {'ok': 75872}

FOLIO
Total: 203
Completed: 168
Coverage (%): 82.76
Binary accuracy on completed (%): 91.67
Coverage-adjusted correct rate (%): 75.86
Automatic route match (%): 100.0
Total time (s): 46.361
Average time (ms/query): 228.3813
Statuses: {'ok': 168, 'unknown': 35}
FOLIO 3-class accuracy (%): 83.33
miss: 621 prediction: False actual: True route: folio status: ok
miss: 1029 prediction: False actual: True route: folio status: ok
miss: 821 prediction: False actual: True route: folio s

,dataset,total,completed,coverage_pct,binary_accuracy_pct,coverage_adjusted_correct_pct,auto_route_match_pct,total_inference_seconds
0,ruletaker,75872,75872,100.000,100.000,100.000,100.000,46.156
1,folio,203,168,82.759,91.667,75.862,100.000,46.361
2,counterbench,1200,1190,99.167,94.958,94.167,99.167,0.159


reasoning routes used: {'horn-forward-naf': 75872, 'folio-formal-resolution': 203, 'counterbench-scm': 1190}


## Public-dataset submission CSV

The evaluation above predicts each public example after removing its gold label and dataset/source metadata. This cell exports resolved predictions to `submission/submission.csv` using the required `id,label` columns. Dataset prefixes and row positions make IDs unique even when the source datasets reuse IDs.

Examples the reasoning agent could not resolve are written separately to `submission/unresolved_public_examples.csv`. They are not silently assigned `False`. The CSV therefore contains only resolved predictions; check the unresolved log before treating it as a complete set of public predictions. The TA can use the optional private-test cell below with the private input when performing the final evaluation.


In [53]:
def export_public_submission(results_by_dataset, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    prediction_rows = []
    unresolved_rows = []

    for dataset_name in ("ruletaker", "folio", "counterbench"):
        if dataset_name not in results_by_dataset:
            continue

        frame = results_by_dataset[dataset_name]
        records = frame.to_dict("records") if hasattr(frame, "to_dict") else list(frame)

        for position, row in enumerate(records):
            original_id = row.get("id", position)
            public_id = "%s:%s" % (dataset_name, position)
            prediction = row.get("prediction")

            missing = prediction is None
            if not missing and pd is not None:
                try:
                    missing = bool(pd.isna(prediction))
                except (TypeError, ValueError):
                    missing = False

            if missing:
                unresolved_rows.append({
                    "dataset": dataset_name,
                    "id": public_id,
                    "original_id": original_id,
                    "status": row.get("status", "unknown"),
                    "route": row.get("selected_module", "unknown"),
                    "details": json.dumps(row.get("details", {}), ensure_ascii=False, default=str),
                    "error": row.get("error", ""),
                })
                continue

            if isinstance(prediction, str):
                normalized = prediction.strip().lower()
                if normalized in {"true", "1", "yes", "entailment"}:
                    label = "True"
                elif normalized in {"false", "0", "no", "not entailment", "uncertain"}:
                    label = "False"
                else:
                    unresolved_rows.append({
                        "dataset": dataset_name,
                        "id": public_id,
                        "original_id": original_id,
                        "status": "invalid_prediction_value",
                        "route": row.get("selected_module", "unknown"),
                        "details": str(row.get("details", {})),
                        "error": "Unrecognized prediction: " + prediction,
                    })
                    continue
            else:
                label = "True" if bool(prediction) else "False"

            prediction_rows.append((public_id, label))

    if not prediction_rows:
        raise RuntimeError("No public predictions were available to export.")

    submission_path = os.path.join(output_dir, "submission.csv")
    unresolved_path = os.path.join(output_dir, "unresolved_public_examples.csv")

    with open(submission_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "label"])
        writer.writerows(prediction_rows)

    with open(unresolved_path, "w", newline="", encoding="utf-8") as handle:
        fieldnames = ["dataset", "id", "original_id", "status", "route", "details", "error"]
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(unresolved_rows)

    counts = Counter(row[0].split(":", 1)[0] for row in prediction_rows)
    print("Public predictions written:", len(prediction_rows))
    for dataset_name in ("ruletaker", "folio", "counterbench"):
        print("%s: %d predictions" % (dataset_name, counts.get(dataset_name, 0)))
    print("Unresolved public examples:", len(unresolved_rows))
    print("Submission file:", submission_path)
    print("Unresolved log:", unresolved_path)
    print("IDs use dataset name and row position so repeated source IDs cannot collide.")
    print("Unresolved examples were not assigned arbitrary labels.")

    return submission_path, unresolved_path, len(prediction_rows), len(unresolved_rows)


public_submission_info = export_public_submission(results_by_dataset)


Public predictions written: 77230
ruletaker: 75872 predictions
folio: 168 predictions
counterbench: 1190 predictions
Unresolved public examples: 45
Submission file: submission/submission.csv
Unresolved log: submission/unresolved_public_examples.csv
IDs use dataset name and row position so repeated source IDs cannot collide.
Unresolved examples were not assigned arbitrary labels.


## 13. Reasoning traces

In [54]:
show_traces(ReasoningAgent())


QUERY : Bob is successful.
ROUTE : horn-forward-naf -> True
    successful(bob)  [by rule: If someone is smart and kind then they are successful.]
      smart(bob)  [by rule: If someone is young then they are smart.]
        young(bob)  [given fact]
      kind(bob)  [given fact]

QUERY : The cat is nice.
ROUTE : horn-forward-naf -> True
    nice(cat)  [by rule: If something is big and not red then it is nice.]
      big(cat)  [given fact]
      not red(...)  [negation as failure]

QUERY : Tom is happy.
ROUTE : horn-forward-naf -> True
    happy(tom)  [by rule: All students are happy.]
      student(tom)  [given fact]

QUERY : Bob is a dancer.
ROUTE : fol-resolution -> True
    1. ~dancer(bob)   [premise / negated goal]
    2. dancer(?0) | ~like(?0, music)   [premise / negated goal]
    3. ~like(bob, music)   [resolve 1, 2]
    4. like(bob, music)   [premise / negated goal]
    5. [] (contradiction)   [resolve 3, 4]



In [55]:

cb = results_by_dataset["counterbench"]
folio = results_by_dataset["folio"]
rt = results_by_dataset["ruletaker"]

print("COUNTERBENCH ERRORS")

errors = cb[cb["status"] == "error"]

print("Total errors:", len(errors))
print("\nError distribution:")

display(errors["error"].value_counts().head(20))

print("\nCounterBench error examples:")

for example, (_, row) in zip(
    public["counterbench"],
    cb.iterrows()
):
    if row["status"] == "error":
        print("\nID:", row["id"])
        print("Type:", example.get("type"))
        print("Given info:", example.get("given_info"))
        print("Question:", example.get("question"))
        print("Error:", row["error"])


print("\nFOLIO UNKNOWN CASES")

unknown = folio[
    folio["prediction"].isna()
]

print("Total unknown:", len(unknown))

reasons = []

for details in unknown["details"]:
    if not isinstance(details, dict):
        reasons.append("missing_details")
    else:
        reasons.append(
            details.get("reason", "resolution_budget_or_unknown")
        )

display(pd.Series(reasons).value_counts())


print("\nFOLIO RESOLUTION DETAILS")

for _, row in unknown.head(10).iterrows():
    print("\nID:", row["id"])
    print("Logical status:", row["logical_status"])
    print("Details:", row["details"])


print("\nRULETAKER INCORRECT PREDICTIONS")

rt_wrong = rt[rt["correct_binary"] == False]

print("Incorrect:", len(rt_wrong))

for example, (_, row) in zip(
    public["ruletaker"],
    rt.iterrows()
):
    if row["correct_binary"] == False:
        print("\nID:", row["id"])
        print("Context:", example["context"])
        print("Query:", example["question"])
        print("Prediction:", row["prediction"])
        print("Actual:", row["actual_raw"])


COUNTERBENCH ERRORS
Total errors: 10

Error distribution:


error
No exogenous assignment is compatible with the observations    10
Name: count, dtype: int64


CounterBench error examples:

ID: 77
Type: conditional
Given info: We know that Ziklo causes not Blaf, Blaf causes Trune, Trune and not Blaf together cause Vork, Vork causes Sline, Frim, and Qado, Frim or not Qado causes Jurf, and Jurf causes Lumbo. We observed Vork
Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Error: No exogenous assignment is compatible with the observations

ID: 277
Type: conditional
Given info: We know that Glent causes not Razz, Razz causes Pex, Pex and not Razz together cause Zurn, Zurn causes Melf, Crip, and Flen, Crip or not Flen causes Tork, and Tork causes Zlim. We observed Zurn
Question: Would Zlim occur if not Glent instead of Glent?
Error: No exogenous assignment is compatible with the observations

ID: 477
Type: conditional
Given info: We know that Praf causes not Vank, Vank causes Scud, Scud and not Vank together cause Wrenk, Wrenk causes Yobb, Glim, and Spog, Glim or not Spog causes Torb, and Torb causes Klep. We observed Wrenk
Question: W

resolution_budget_or_unknown    23
formal_parse_or_cnf_error       12
Name: count, dtype: int64


FOLIO RESOLUTION DETAILS

ID: 1015
Logical status: Unknown
Details: {'reason': 'formal_parse_or_cnf_error', 'error': "Unexpected token: '⊕'.  Expected token ')'.\n((-((((YoungChildren(bonnie) | Teenager(bonnie)) & -(YoungChildren(bonnie) & Teenager(bonnie)))) & WishToFurther(bonnie, academicCareer)) | Chaperone(bonnie, highSchoolDance)) -> (Studen(x) & AttendSchool(x)) ⊕ (((YoungChildren(bonnie) | Teenager(bonnie)) & -(YoungChildren(bonnie) & Teenager(bonnie)))) & WishToFurther(bonnie, academicCareer) & -(-((((YoungChildren(bonnie) | Teenager(bonnie)) & -(YoungChildren(bonnie) & Teenager(bonnie)))) & WishToFurther(bonnie, academicCareer)) & Chaperone(bonnie, highSchoolDance)) -> (Studen(x) & AttendSchool(x)) ⊕ (((YoungChildren(bonnie) | Teenager(bonnie)) & -(YoungChildren(bonnie) & Teenager(bonnie)))) & WishToFurther(bonnie, academicCareer))\n                                                                                                                                                

## Private-test conversion (for the evaluator)

The public development/validation predictions have already been exported above to `submission/submission.csv`. This public CSV is a development artifact generated by the reasoning agent without using gold labels. It contains only successfully resolved examples; unresolved examples are listed separately and are not assigned fabricated labels.

The TA can supply the private test file by setting `TEST_FILE` in the final cell. The private-test conversion uses the same automatic router and writes an `id,label` CSV. The private test data is not required during our development run.


In [56]:
if TEST_FILE is None:
    print("Private test file not supplied. The evaluator can set TEST_FILE and rerun this cell to create submission/submission.csv.")
else:
    if not os.path.exists(TEST_FILE):
        raise FileNotFoundError(TEST_FILE)
    private_examples = load_local(TEST_FILE, TEST_DATASET)
    submission_rows, submission_seconds, unresolved_count = write_submission(
        ReasoningAgent(engine="forward"),
        private_examples,
        os.path.join(OUTPUT_DIR, "submission.csv"),
        TEST_DATASET
    )
    print("Inference time: %.3fs for %d queries" % (submission_seconds, len(submission_rows)))
    print("Unresolved examples: %d" % unresolved_count)


Private test file not supplied. The evaluator can set TEST_FILE and rerun this cell to create submission/submission.csv.
